# Phase 7 (PCR) - Ensemble of the finished runs
Rebuilds every finished PCR run (Model 1 / Model 4 with LoRA / LoRMA, and the distilled student if attached) from its saved adapter, embeds the dev and test cases with each, and combines the runs' cosine scores. Everything that is not fixed (member weights, centring, CSLS / dual-softmax, BM25 weight) is chosen on dev.

**Attach as inputs:** the data-prep output (`tokenized_pcr.pkl`), the Phase-1 Model 1 folder + `legal_gqa_model.py`, the Phase-1 Model 4 folder + `model.py`, `moe.py`, `legal_gqa_model.py`, and the output folder of each finished run (each has `adapter_best.pt` and `run_summary.json`). Any run that is not attached is skipped.

**Output:** `/kaggle/working/pcr_ensemble/` (`prediction.json`, `submission.zip`, dev table, `run_summary.json`)

**Combination:** each run's score matrix (query x candidate cosine, own id masked) is z-scored per query, the runs are averaged with the chosen weights, scaled back to a cosine-like range so that CSLS / dual softmax keep their meaning, and optionally mixed with token-level BM25. Candidate weightings: equal, weighted by each run's own dev F1, and leave-one-out (to see whether a member hurts).

In [ ]:
import os
import json
import time
import zipfile

import numpy as np

RUN_TEST = True
OUT_DIR = os.environ.get("PCR_OUT_DIR", "/kaggle/working/pcr_ensemble")
USE_ONLY = None            # e.g. ["m4_lora", "m1_lora"] to restrict the members, None = every run found
TOP_K = 7
os.makedirs(OUT_DIR, exist_ok=True)
print("output folder:", OUT_DIR)

In [ ]:
# Code of the finished runs (config, model loading, encoder, scoring, adapter). Replayed in separate namespaces.
_SOURCES_B64 = {}
_SOURCES_B64['m1_lora'] = (
    'eNrVfYtz20jO57/CdSp1ZCLLj0xmdlXjuVEcJfE3fsV25vHJKhYtUTZjPRhRiuOZnfvb7weg36TsZK+urm62NhbZ3ehuNIAG0Giw'
    'v1FMy/liGc2ry5n6Wd3b3x+r+cw8XE/mV+Zhmi1vzMOymObmYZHNRvOpBXezWhYT81gWw9uJrfxnUY4LejZvZqtpeR9lVTQr6e2T'
    'qDf7HH3OFlWULfJoPpvcR+P5IprMh9kkWubVsphdt6OTWfRLdn09yaNJnn3Oo+VNPo2yyXyWty9nr7sX3fT1wVm0h3m289nnYjGf'
    'ta/zZXy5cbp/luryy41WdIwmyeXs6OR17/ChNqZCrdHpHw+3Of3DaXLy4eKhXlQxNbjc2LrlGW7dzRe3mPRWOVyk0510Ml9klxsJ'
    'IUs6OO4e9dJ3B8cXgHq5Mcmvs8nm9SfUiei/J0DfZJQvolk2zaPxIrue5jMQwJhwFp3eZFW+uRMdzUf5JNpRde3UGLiFmwJuOqW6'
    '7fL+coPGcH508ksvveidU/dvskmVR8F/T6KLxSpHKZbuPlqsZtFyHg1v8uEtD2EIcFE+G9Fb/AFhjJuRwz1h5tEejWfncqNzOSP4'
    '3gioKx5Wr/caj9/tXs7OPhynF2fdg2NTzG/c+me98w88Tx5p039PmJKjVRllMvZyXgCN48UcdDeLsuUyw+tRlGeLSQFsY5qXM+41'
    'PTlOX/d+bcaOwQ2K5tFykRUz0Hw0yj9Hn1b5osirKAaSZoQpsEMBNpmDOybFKFsW81krmmTVMsrL+fCGCldVPkqEjV7Phyte6UVe'
    'LvIKv7gFuONkP/31oPcbr2q2Ws4VofjDMkUxT5EWCl1mm4BVRp+L/A6zzib3VVElRKs3eTYSql1m4P4N/U6eIjDw5cZ4NZkQzbzr'
    'dV+n++8+HP9yjjHsAE3dg0PvxZsPh4fpUfd3+/KftQEayeDNR/fBDdPDHq35zvbud00Lupzf5rMqKrFagHOXLUZRmVVVKypmw8lq'
    'BJYT+pzMK/p93juVJpez05OTw4Pjt9zjNM9mDRhkFKqy+Wd0waBuVrNb3S/jhJaPkNzYEVbyw3kvPT07+a/1pPlEqCa7giwcRV/w'
    '/3Ix/5gPhTyqZbZYViReqYdiBDIolvdAee/otHfWvfhwRlS/3d5+KVTTHWXlkgTA4clZNyU5tfO9eugenr7r4sWLXfXi9dnJKQSW'
    'aX9I1Xfzze+idf89iTKBT1J7MaPJLrIl+I+mmHL7nTXtnzjTClvTwC8ICXh1OXv/oXd20DtPMb/0/AK4BPXQkskbKiWB/aKpB+B/'
    'WXwGz1XZtJyAm4k2iA/v+Ve1BO3H1xCRUZUvZYOarsB5k2xxLVQUVXNIWcW6WL933bPX6XHvrd+38/6se/yWliB+0Yq+207c4dwQ'
    'Rc4gdmVMwznLb2ZGsHo1KYa5FuOvjnZfQjLMNnlwkwIyIa5ui1KW/TNNYDkvW9HdTYHhTrP76CqPVrNJdpXTNBf493M2W2LAvdOT'
    '/XfEcVjQ33oHb99hQ+rtd/+QVca77tnRh9P0zVl33yz82zNi6MMDQvUOVRLuOzrYPztJX3Uv9t/VOPiJ8EKd+6LRasGcZ9az92v3'
    'MIBFNEmcsf+ut//L6YlsfTX+eBJdL7JRQTLQSmyAbAGp+QjTBiVlgBW9fRVdfIcto/trL+39ijViuiEk7BJZn7wN3r6k7eX8/ACC'
    'fb97mr47+XDGgmu7/dLjS6Cedrsqo+WDlgQCmpeQL/QTIhniuaIxUL1Z/mUpewYR88FsDGk/G+adKKfFw4IDJVir+R1J+eHNHPJc'
    '7RPg5ZPT9Bf0/8Pl7Pjk7Ch9e3ZAW18f2yZIIidhvA0hjcdhNano8aX/uKOKR9WUK7e3d8MXL4MXO8kARPzHK3SVHnaPXr3uEgb6'
    'qEmluy/pX/yzQ8+77e2BYOQuL65vRPcg+bbJZBvTPvGcNooE4oHWBYqA4GS1gM5XDecLMDnVZTmfXmC7P2aE7/5zWxX8ssO091I9'
    'vmLa/OGlo6DwVqxkA63sd6xoWN1BKRPexrH7T3lpmGJXnhtEzHdu84DylZpSJ68XUlKnMNVPCVYg7cfRcAoimY6oUrQhA72y/YOu'
    'MN5NRqxirlaDokXbZlupj/snxxe94ws1WzvzTSiCdpvNlVwEuaHz27xcsqALdiyCB61tmt3mo2JRxUqTbUX5F0ijdH67R/yJbsVc'
    'aFdgwZiUNLyale2Gt3ry+/PZuLiGAZCBNmjoIMCNUT6OxsVslGJ8KVkU8WI+X2LGpOUmai1vCghpUOVAHmnUVAv7O/+tVDVdhnGn'
    'JcycFv8iQIBHsOU3NcMM77LJLXeWOM3pP1ATq9ioZhsFdfSo2llZQteNAY96bH+EaIpt/zyJRJoC7CSfxdSKNd9tB+QiX64WM7Yt'
    '7IzbFcyq+Da/35uwCuFUpOL+9uCSxYxBIevzKXqPqd8UssbMbTxfQVb5KPwGNFkbppiVKyg6Ls5ImO0ZaG085os4seWYOEQUL36b'
    '7dIN7HBLH72iRFnpjjooB6gA7wADjlnlHnQzW9WG5TKUtnluu6qDYpTo9dOj99eKqzy+WIRMasxToiadYPJYnWwiU3JRFAxIwS3Z'
    'QHdeMEiz2gBozGKwMY1CwXn14fj1IbTM7gUJK5+p+rUVHLCOT0Km+DMfpbBJ2+XthAVKDuumEaZH5XoU6+FgqG57NVpaaloUDYzl'
    'ShU7NTVioDnAzHqD8R/Pl28IC73FYr6IG/pjiLKiUZdNOMcq/sGxeOarJaYfxd3RCLszfiYiRLWQoprRFeBMgAIIKHdYCv3WxeDj'
    '33U9BPwY2PeWytbBenj6x/M19n50V4AMHV4LcGJ8CptMkv7UebgKEM/dDM6fOfwkTRNn98ljVNfgDmFS9NwUIXpqHT6MnbqDo4E6'
    'ClKsc/Y3RaS35IukCRu01Tq4OP0jcTxey/lieOM/tWczdoHNaq/b49WMjR5oQqjx5nImJgAXk6etajvuCNXYeUOWH9VNHSFJToDe'
    'rwf7ZHcIIOiRsCZIJVyNyHFUjFUBPbeLKs0+Qz8jGzNOIuJzkrrlimdOinj3iHQgAdpe3pe5+GgE2uUMxenriz9ObYdjbOdLrcaf'
    '73cPe0T+CtTlTCpNs9kKS9KkFMiAGcXSK1HIv6NxCXWeHBdD2NVcqkAy/s8vumcX6cUBe3rIh9mmf2jPcdX58x5ZNnUF/1n04vtt'
    'toHWOqh8KOSqYofN0iXCeleMi/grIbL2w+w6xw7kij/CwOKKOsWaj7UgBnxxwbZJgYrHgojeaco2AiHzFditysu0gANngOJTGHNe'
    'YZmNTOGvJ/vdV+n5wX/3nAqfge6rtIJglUp6kapyUpCWRfKwrx/VFvLviFtxoQUqJeQAKUZcJCOV1xiYfi1jlL1Cj3dPVe74ujO1'
    'oo2dgFY3ZLLDXQcw2DBH7OPBEmWTu+y+gl1cQXuNru5pF79mRYzML3ohanUx0qrz25NDQhFNi0xuTErenffIFP3rb7ApOQNoxrR9'
    'e9PvGDkl5cwq5Nk2Hs26ykLQPhUjgkXd9LnlwKmuOu/HXNKiyskAQ4GjInZa9PF+oDyEv91kS+PYg5UKDMSMNeCgmiuH1RxsVinT'
    'cwjMXeUiXdjP6M0RU2SjXaQ1Wab1qSqtqgkd9TkPIXk9BZ4wna5mZInIxJKvRI5A0kobqWg1lCS+LTAMgHGlQPGyw2mDlOKhApEt'
    'FhgebBr8yO5j6VuVlfP5BIU0AuYsPIK1CAuqn8Qn3qdVJ3o6Mq7gfwsAvPk39+74p8jVGD1t74zxa1Tw7238zr5IbXp6+lS7ixgK'
    '6J5heF4rOPWjp7D0zUQVOdGIMR94AahtiybZpi5jvMFUpU9Vg8uyL1S0QwITkpOaW3Ql0RaDSZgQyZmckjtkwov9Sok/9zWR1F9/'
    'B+pWgys6itmjcFQMF/PNNzs//6B8JAkLDReiiA7rOra+cLXIV+BHtjEwJKuy81t2SOD1Jvu6NMVo26//mDO8ZdzUg07NHiEIHkJI'
    'dLkv+lRrEP3kDCUgS2+M9aYNldU06Y/hV1tQ16HcRu7MpIbjjzcVnZXTxxK8cHReIZu4aiSSnkAqHwYVOgcGUk7dOeXO+YGUi59d'
    'hDhVMK4N3j2VkiR/JsVV2zuwLGZVCTczUSZORsXWwLscNjV8Wdr6gHZO84qNfgeCona0K3pw2/Q2JZ2NdduUzjFJnasrnIG6KKUo'
    'qYPEyxVAMVACH9M/agC8zecLmEfoVirGBhTz2+FRi50paQH5zcJIlbYP6dfb990388V+tqqyyeFRmzsBp7Fohx5mlPuWsolSgoV9'
    'lMFpJ8+0qMgxlMILoQU48QgexahWvStWd2sTFvqDxG4c1OYffAA5TYku2uJEBKN6vtbC+g1lr86nVzlv8JZu3X70ZoDfjlzR5pH0'
    'USmnE+SnakqjEYWGJJoLLxHKW8GRS+SDRl5VcsnH4bxtZXfqSSLCqdYDmH7bM3HPsLNAgdUGTDj4OzoxPD65iEja3Ku5tPRUOhhv'
    '9Bwu6YXXS7/zcqAGgNVXJqFV8GgprLoWqFo4GzjGsVgnEiOIW9GkmgARugCLVQ9T0UJ+YAQ/BopeIy4Y7pY7EJo3ghhgeW0o/UcR'
    'FAm4B8inHR2Myey7yyqtvohmZI0uREescOIC1YjhYFFxNlMsydGKbmKnHwJAMk98kWCqAucu2E6vVqKJob1xsioVTKaQf4IlVHlH'
    '9mKlCqLq3MF81kR0DcymuQ3o9sGAdfHubplrsN5yB3XXI2jzpwf5c0OJJTEKSak7PJKel/NYrDt+1yZcxUrtLKkfvCuzBeQwjhQr'
    '4xiDNxnoKrDrpXQApA/eqZNjOOXPyXTB4UtcthF7kgNitA4g+notURzK2vGw0YY5Ueb9nUFo/jPXkxWM+kk75R0wTUVAMHjS6tq7'
    '4yMIlpsCRx44IyLKfjoS7UuNcguHod9ja5QRiLJEI0WswUJpGlCzUu2dTw1GK1EyZqZIvbjC5nOr7DR6ptZV/imFpJEXpH/jICq1'
    'OydBGS6/WF0FS8DxCiAmGkdsWLTF4zKBLkp6F6A1nEHjKCvmlnS4vkxEq2FIP3k+0tCLwsOJ4qdV0jHYkW64deK7i+0Jxk9SHnpK'
    'mySFbfSUWRQxRNd8XJ/NHP7TQ9GDMK0aRnK1yLNb62O/yoa3V+TbkqWO2aVFWr+m1yfROelSON9FASsH+rxes7o6ICWNmo4QSzJS'
    '8zFpeJoHI9mwZbStaMEc6Vi4MHynk7yq2tLlBdOzGoehYyn7Irw3WsxLRe/OkBNtmsHnj6WVqimq5ikfSJOOdNFyYIujphV9aY+I'
    'IxyL7Wpyq1iO6bLylWHxABXCwmnO8Q1AAFOOf/QbLDKhHAepMax39LCH/ycNhzDKR45SqqinI9Tb4qNk/ZtlR8J+dBfAF+3Kchxr'
    'MfrFTFsk4qF0Q8vFsdZyTyDY5o6f3IVGY/nyTWNRkyCxPUvH8ZfkUp0fH8EeZN+ZOVrnDSq7xqDE12so+3/gDOVupuuBPKRYvI9z'
    'xr4VrIv5VW5ceEsEsYCD7OEd/SEsvFi349O/e/EOAh3+mYCUhTT2jISnLjAWKJTXcnBn96M4ZCIeSpLYLUPwMXYbW7Ldk+ptKfQb'
    'CVK0Fy62A4D9fk1m7OYOH+VY4E6BGYAyhhXMUTEeN4HEyaoFk7Szq4pakqnsbiH+yn2uIu1h5xcdiuSATOBxsyX1tP1iTCoqbH2G'
    'LN0/bX83VgKTarb4LfWDIAsYM2reEL/tf8le4PU6mueym7PK5AhCVeFyQ5/Csq2VSpxRfNjiappg9mnfdE6ZzbZwSFrc/mH9dMzQ'
    'NrYdDpuSfZeiT9vDvJjEh9gW9w81olV4k3URATYNQAx6sbIDI1vBRf9BlFntnA+bLB8VZ9gQYtWuSZbIILRBMcUZVwGXxz5wcYgF'
    't4Ndw/vsf6GOwEkTsj4zEaPSuBUO8/ERsCSiU4u4SnTfQb8z0UhxQOm+Y7Pae6fRGfo1rM0fOsUMZMdoXwevyTtSh6dG5Rj56xZU'
    '9f1cNfp/uaBeRzSqr1g27iOpw/rowuI4mcfHDJlCY40/AhuQXzL67RrbUKgAtlXypcq7xA8YkJeyrYjKs9/Fzqtc3FrDgS6YijdG'
    'PBFGr1EeEe2Tvtz4pGNBh7TuxShx/MM5+1ydXurSwSnsUwPF9Ao8TgVh3sLBBJFPc0MF60UoKI5hT4bgSoRRpdz4EpVpPbF4pdyu'
    '9E6gDhqZ6UEY8JmP6hBU8JsfWMFCwBOn5AIgrcv6x9ylFyCGfEZVv+pUWG9H4Gp/cg1x6Fr5z9zl1q/0wlI0T3pFO0B8DYlS6t5Z'
    'R66WVlYYlzlX8yUuTWJIzgvVKiBeC0sqWgVVlzynkGCLaIhJEuwxY0eGBS0p0YdCoABSNPdQDYLw++8UQDFpVPs/88W88to3tmFG'
    'bsnEcrIVKdzUxwSbzcHY1VD7aNo5ZEy3cRpJ5mAA36nI9cSt4UTIyJj7BRUeWjyo1VInqeSg40sUTCqOzdyq11AQE8+y1qvNpxFK'
    'v7pp6d71TGlFbgJTgU7fdCj0nolmdvl2flcZhTETGaaBbA+MCqgPoJMax9/0CYQZDIWmDfRRirJtQ/gXdbDt1awCM+d/YmNVfdBZ'
    'H4kmC+ZH3YlTmzQ8oOrGM17gO8B5BzW+gWAlQE4LVgrJu7ATSFJptNXQCZr4ndD/hhMKhj01Mc8xQgGO2J1rxauEe0NnQDg0xlGS'
    '0bpcE/LdivT9ArIIdR3xOpUTunagjU0iA44J9q8NaNsqhTOpWKYpNo3JGKh2OaFalRSg1DZ1nAWl6tathNmcakdLLMsnTEkS3N1f'
    'HzBH1uy3zVut7Z3YjTmKj2/sxLSRLfP6ktS3ny/Rzy6gNtsQ9D8OX4eX5sNhzxwXgTd08L6C5NdyllZ7eNZyJRzEan/VgZ2ECD3C'
    'mZY/XKYNqcWItWf2ASrlwQkpJNWeYDiT5BZO8GA2vRplUdFxYEMSJYl2uVfFFCEhC8Oayzn0ANxSIWYF6Srng+I2OpEzMaZueCzG'
    'wk46DioJQ70Vpxc5nUHbzZJfgMRG+ZdwCyW6tqQALXrWks4Tb8OmdjzfPrfoSLvnUtWx9FnWO3145AYw4YkwVdcbskWZJ+1bzlZU'
    '218b6V7Hs8QizVLeqpxYG717mSgbBN2Kv2RPR78E41ROx726b2pUOWNQkmDP3xn4j7Z77Tbh7fkuqddDYGrwneqxvGwaxRtIgMUU'
    'd58gM+UlGbXTvc0dtzYTi16DGrAHFs+hKqNTaR/BZJKaYajAKMxe+iK9WvHVDAZFZT0k+bRc3sczvT7ykrQav37f86c4w2hoOAh3'
    'uyDC1460r6CTULFv9Y7oxACz2ziFeiraeyBcapoqvBksWFy2u20Z/d3qSQws6TTEVayxF5r12mFgEXH/5vQucUZJDnBFrGsEpjOD'
    'dBk6svh1HeVGmzB9+MoKb1sGltUsZfotOzSjOz0G0egXDKzNpJBShAmdP8vYHbD++rs8InAcHvFXnE0UbZrRUYAyd/bYg6/W5JPo'
    '3LyLvFIm0X1jyEoxdoGs4XoFjv70O051I9j93tbGx3zic+/QgPpExEWw3R6lpiYYzxRFlJTeEstGgHwmVPoAy3UA2aQtNcBH9Jb3'
    'JCUczlPD1C7eNoIrXeXpNKherq+uCOETbzUl//seW8Rprp3DN4jkgrfvXrM2u/pUsBddK+LAjGtcOePoIh2v8T+/IqJtln5qWpE1'
    'dNPv7G5vDzyn1Cc3RMuREvVV02iGF6dpzZqJp6HH4aM96mVNGkKznDtsEoVFp3oUpYUXZPjTLd1cXsaMVhxkiU9WgefQKTiOMfXE'
    'fRq6Jj8HrdBdwYPjN8arfLmxiUNddXbqWOsc+FPFsuQtjwyGWEbr8HoPUO9VvA8F757a2B+p6OBpuvL2vf57AB/QvifjhVy6zfOS'
    '5IyK/zBkHmzb72HATVcN2/ZpUPG0qSLdjHoPHfy0bbz3MN/S+Vj5oxybvfT3otI5brPN+iVtp4VzO6glFGsbfgoaAjtcQwAEG9d5'
    'f9HSoD8NCLZeNXv/VRFKwc4eOhKg0xci1Y4KNlVRATkVwtbwmPrcLjbIdnKfEsJiBD3R35acLTuhM/RWDHO5B1i3aJQrdUzmmtVb'
    'ikpexOdJAyy5NVi7yHPelvBZCnOaxP9LILQ4SM5ZZ7K52aEKHpLhIsbXOARc+z8lBmIv0B1UmPI2vhVqANHxiWv1GO0hPIlu0nH8'
    '4TmdvDDEGgpQrXnkau2aUMDXIB0M/Enzh20vrCnzCgeiNYrrtJqPl+Sg/VMmBPfs8/Wl28l/OmI57f6VcCVn3TR+R1z8yaIiJf9K'
    'bE4VHqMEjmaVCxt4LY6OYAXacF1MS3Kb74lj49w/+sWQWbtZQzDbhl44pHVPNWjsCijnAamxsdgSz4zqZZOBkP9bDbg2HGQaofqm'
    '7bNn0W7yFZ3JosbnpostiPgM9vMn2M+0orjI/r2/JX/Fuum1AcHT1kWsLbLbHGJrBrq1fLNjVDC2a889dsESQIGkIG533N6VSGsv'
    'A4DlRd+N12AEf1R2VH8xQJestNXupoF5tSlGOt/Hgc8XuhD1arhyNjayyhGEOMaxL5BHPVV6a6OoYgc9y9I6pWcpxzc7z6VQx/a3'
    'SPxrFWtOsCDVE1d7YPjPxfty7ZVwTyi5bTB/7CSAuX7ndlC/SCpmUoMfC/N7vqf9wDRXvNgixTuWPhFybRA59IsFU7p4vKNv9Ksu'
    'AZjb+AE5XG0X/FPi/1QMMlc1/QVDcgG8w0XQHbtueEGWJ0Tiua+JBIvGuGCZU6P6x1f1/94qmgXTa/WfrpP21zethejkm/hP3YlH'
    'JgYcs3Mcu8kbYm7Iq9g51iUhzyN9oXGhUhrA5Z8vppKZQl9Z4Vh3lZtpWJT3iEjOyEsBB3FVOmFKU9ybJ8Cxg7qbbELKVe3u/daW'
    'vqKuDnioDR3BhjXrKgdp5C5WoPQiVARqJXxWWLg+m4bU76DFhyOb9LszGDjSkcVwipiARfEl5lsWZsCQdeWS3RM6kEBJP0d+cWyo'
    'LwPJ7sRKCixnzIzovQA3tWL60+ff/9CHOIPaERiV1+4E2yHbw3t66m/uDJLHrjDzUrfsNRlgEtcroF9yXy2F4FTKQ+1IoUV3y7Bq'
    'x1ROfUKaccmo6zSmNus9L3YTH37jrCA7CBvcX3DFXQ2phiT1PoUscY/vttec2unRPlRfD7j5PDfsz6VPPch1fbl1qSQ4BUKFaqFJ'
    'N9ZNW26fLXt3SNBGsVK8K+/F6hpOxvHmTrR0EvCxsgLtXTAlejWbnKYC5Sv8HbqqXsrHzrlLTbjvm/toctK8/7o+nPN4I8J/p/NV'
    'Vx7o8ZLqCVzHxlFAjhNU/t1oOEoajGXhrpD1ieDEv2uFiU4OZuLe3vvd6loh3RvCKzQoqPExzB6Qeqy63aRunlNOFFYUzYOOF1PE'
    'k1Wy7r+zAprhhvseHScukEAGYcprO84+XzME2ltGE3Mvy1peUK9gk6azP6UbHO3l2JSZzFRYCY1SPBAUJiY4YNJTG/CYMSdCYM0o'
    'cCqAi8l7VPd5pJOyPIsYE5uRSsvyXP94hkn3zbgGwArPQnvP1BWGPUJq36zIAM0A/pnc8yL4z3mWaM29S+PfOPjDYzQFr0FktSID'
    'HbyLI3L9hqSXvDDsp4ggsQ4Uj/LeN1Pe+7bacIQydspY3jhOjdo5fUxujt/aFwRKSCJZL27///CBrHNkIGY2re5gaDX6rERRYplm'
    'bFR++m/yjzsWrKpi6N07slBnoW6wCtt7HIfGLlSJs3U1vPNtfUD3uEvN1xkdZwx1YHIyhZjaCabgunS2fZ9OQ1TYRKD7OZgaNFHa'
    'XDP2W0DbbyjnoeyS3bhTL2yIbgva0H4O8M/UqjxUd9e3fkMPw04SejC8MKHQRtjVa8DprxpakEXSEgPGGpANtuO69rQmekP663KD'
    'iYhunfEP9gzz4tMr+cW3JbBo9EYWj+7OY/noBf9ocYwMnavTK/xqNePL/AcAMKJwiYLS33RoSrjjz1YVPXKPY0qEiWn+3TAFIXzH'
    'EKEphSf8iCiON0d9gUSHU30zWee0f0mHOuQPYYMEZ//Ij7dADgK+rjBGxqelx+DUk5dmiPI4PMBH4+uQy7+O/8bXfbsQWid4jLOk'
    'lSzVQD+qlRo4SqkU6AUb1HlISzOFXssSYctnD8gqBeObuMMEh1DkTjreWSs7DTa/ApnuPYW1JrnLLWroX8FdOlgcl2pwgoXrIvDv'
    '0GUHELSOSXLCmqQ8ulsUS2jSlH6B2nX4PFSuqtPVioI0tAq8INlYFrkmQaxcTGePBnLEqu261fuiHGug7ZTuCUhkEFSWUvtRZ/bY'
    'nx7gu7nN+baFG0jG1X60YNj16d96MWFEEjUGrcXUtkvKKIIdBxk/TWM5mwZvcEopgyKiEZwlSKmcNWK7vsJ5P7CX4URBAaAUqZQu'
    'pr/JABScwVcdUQpw1oHSOJy9FJJjVRzsDBc4k3G6IWyH87PuIYeoNYWw3QN1v20DJdCVs0l5Q5c2Gewr5FpCEJYaC6pgHn/miKXg'
    'vK9dvgf0SvwccjN5bYTaVUbbO3DI8FuavL4pbo1gcLhMlft7PetYpkrTbUSzHa27lWjsyBSjwvDSYqa68u4ZBiNS1ehPUEKaziJ4'
    'J2yyFxkUB+UlMyXBE+Q4o0ImYn80EqsQAKCk1Gm3ObBPwmAWMjUT/IA/frBFgxPBgn71YMygYG7x7bApwnGZuqvmRgTWSqmNu67I'
    'l9vRrMa65ObOFh8A0EzJAnOesNG8EhFWtfTd3lXJB8Q2GjP/kg1xb9i9gnGlUvCRTNriNOxyxmCowJmTLzuchWkZxvyGuEoH+4zo'
    '1MfOA/GS9+xtVUxhJCVfdHtBe+QX/wqWLfD3hxfQ8RzyVA9GrIcnkATiZ5cc6ZjYLlXcboNGXMdNPmG7jFr/7M3WtFOtmMAaWsrf'
    'Zw6P1XaXe0g2rkbbwb0X4ktIJAKrVlcqUQOllaQbDEtKCUC3hikD0Z1K42Bu4am7G061tsTyXG5IeiJVK+eTOoLp5huU+EgG0u/A'
    'IedahLqNvtwrL9gI0dOvGooqcux5Q9V+7QtKjIwUUJLGVocqT7J7tr0qBCczH8gF0DaihSQg2dxkbnEQkb7kikRir7unF0i6ikRe'
    'b3sX505qCcGWdK+uldKrkRqRlcf+pWQpbZFskU1KbpgKUpktiVFidXe6avspLIPhaGuB03eaHA5BJc+92XgjGWn6XERVKgsn/k+5'
    'qBSq9k1aV53KT82d82L6OeloC0bOFFoKBdLkpQgHp9KokT4fjfM7rhfU4QwRtLwPiBC+E5tKNJcXZNfvIyvwvyhVLaW7bkU//KBz'
    'fA0GzTdC1UXneqyq6cNewhzSByBESulBS1RpSClEIcG01MjvFhnfr95zFRcQ07XHqLzCuLXBSdLVX86Prn6r1OiWZ5y2lH9FqFX1'
    'pWoFI9bEZCs9iPFsvGQfx+NYAmGqOwGpf09VYXpTYK27nqobq8TCdNUsQk25dJqRcCGXc3UzX03oljqChDpy18vp095C9YfyI6d/'
    'Jxpkkh3ekHdy5F2FV4k4VUfyCQqNudPuWffIXeqpu84Krx0f29LG3JtTO0fyFZVeJZRPQlVZn1LCh+Eg0qbt58maxBDIYkURYzZb'
    'F1+8VXFjiPj6y3Taaf0t6bslyZYdzFYkKSTccwGVeZ+ulsGe1GTDT4HrUIlSGjw+1xKKiFaIUE+RJnAqRRVsIo1O6IyUPTNWgYqa'
    'T9GJ3nGbK9QuIy5zOyFObOLPiv+1uYv/j6YCS4FMQWdG36Q8mamL/cTbv4MODvdNGpu8Uk2unCavTBNtSZNMYQri5FGxk8JFsqSs'
    '+wCD5iNtZGN+OKKZj8cWaQ9wjQyT+23qwn7fQYOfz3xdhwa3ZsTfMADdQJQMc6sJjhLiIHPsJZF2JDol+/v56eGBKA0mVeGAt2nv'
    'Oy2SBZrzF64LdHXBaamgmpjODvCJDVcWmchZt3GnlsBwTbBs/UplLYNhELlhx8kjCeOVKT9k0pwC3x+/89TvNOTSHwQSjRZCHdQZ'
    'LcOBoTQMJzWoiw9eU/1tDCcZIH0KI8WnMEwiQENT/KEODiPHcSnW3wg2M0063kCBTSGJFyHyzEEKKcwqFT1DjsWLRRFX3hdEWial'
    'oQkMntAFJ/+7HuaQ8aaol9mLjHS5hpIueGkMDL0YdBBhKirR7+rU4UILavFM+5N556bQSRby6yB5As3KgAhJSlX3EWRqC5bqHzxp'
    'BUA1vjgfJGSCWo5a8lJ18si3CczZjQ7qVheu6zHeSou7KfjTYIilxpgRmRI1fIiF8Cngf4xeuvOUl9ZtzhqtnbgkRY113/7NJ4kI'
    '0lPjCKI9vvAQ6WgiGtGj6empUngDRkdakQOByh0uwIdpkFWkqpgFzGFd/ZYEndI3v9S36bQiYL+FE0TKoAMJXSZZW8sTGvKbAtXS'
    'tFPn16++qlF490LN+MLqKGhM5krM/bwJ+bUkBRzxH1470ikKVIYzi7OG4zFbaC7b8x1Kg/xkTRtv/iqpo75/pldNL+pzp4n6Sol/'
    'R8Q0krNNfRpN7rxev0MDUqA0q+1LkVvS0RkWiin1HaP1z6hHDhI6dHe+IuXeAOFgHiZE95JGIwqwHqnyoIs+Bb2vousQLWrlO/Gu'
    'KGWsPfL2mokj8NF26mRcE7E94/b45hF6/jrBZaJkm+7GOXgIRYFNXtHfHdQDDlPZADwJw01YyDQQo8YwXXv+OKi7nV0GEY7CldX+'
    'zqB2V7MyYBz6rnENH/nLR/WC9dU/Sa0levIvvTkrqn82VyQwE+eSJUIq6EtKX0qp6x2i6S7pGkL+r0RFRDfeGJC4SbqEQgH45WRV'
    'xbqnTSZ/PU4YG6JYgh2k0TM7YmVi6wwAnOfYL1IlbpC8DdNRAl53oSJ5RM6LmmRMXA63rtmWfJHWM4MfvYzrN2GwToMwHSLSwPxy'
    'etH4YQ/zxR/Qwm25lE9sLFkjfgXl8rE22pKjnMG63TlOD097r9Ne9+zwD0u6SmPsIdHJ63MdxvzuALX582o0be7xzY7J0szP/Bkn'
    'fqVe4IMAF/bGvgkvXS3Ityk2JRwTw6WbAe1jqm1mmxb6q248e231NuvcoP1MTOii3u2/zQKqfqhkYPZvifE+rzGkPfr6yyCbIgcC'
    'vwAHLQAqhxsY6H8HZrcdGkUsxnBKXtncMDWznIr7TqfOldE1aGPdTDWT0QzWYNVFWDC02AfgWA04h4NTDGKFjs6NxeCbp/b+q3+N'
    '0myy2ueuzb4g5uTBo3m2P3SSfMdL6Fr0tfB6G1wPxNmPmhKqQiPLUdxJN6xSkinyldAwoVqt6Vbt82YmCnC+pDAOAkiagnwU7VnY'
    'g9LAkXDSVCUf4g6n/ozd7wc+cyGG9yulLd2kZKhsKgY9qWTgBEOKJPGqA1OfV3F2BVZl/1KRHnSFLXJlKsfmMEccnv3tG0SOiAxN'
    'IgHshAlZ4E4rC1t96FIH61D8zJTj943+Q2/a+Azn9DfJCQFqkYNX0Oswu99zP8qoXeRw6aSTRaiqX/OnuHQPIslTgekmc1eN9Rw4'
    'cMVhUj7WsuPDrtV+C4fXOb82X4zxU07IB12808IJhAE1iWlhgmhC/q7mjw7JNH/hiptKPrMtp7InYZHUsGJNletuOtUSdflix6MQ'
    'v0rt9A4fOuRI3X+BVunrjiqc9bmcwyJVpzBSSbnbdO+JmbcR9pcbwhkggG0WsPPK/Kae7UMOU3AkhW2pmktAGYm9vzVg4n/+YrGX'
    'XUHOie2upGoGH8wy+3fS9GEzytGiKzyS5E5qf9X3wkJ9wBmdwFmfI4RkuKPr0SdsJJIF+Y5MMv099TUizSqIPkWUSC3xqtmeGrav'
    'vko6bAnfBNEZDlqzv8xpVmErYZt1TaS0oZWiGF2NPd4bgRlgNR1VDZ+DBX7uaxWtEqQq8vcYOMyvoaLWjty6imybqmvdya3ePF6z'
    'yv9wiKspRxN/OV6CrtUiW2IN9R7ZJOQrqiMOcqLlV81oW1BDbykPvmFB9fUhYTtbKs8DT2bRh1tTol1P71IU+ZchmE6jxkjdMGV0'
    'HAoKK2g66Gh68SrUUWRIoiMD59yZevU7mjI417de6o4mA/PWyCK77I1duUvacRbdUV2FLQlPTNnOgkFGIhxhOS09hocswuWDCWU4'
    'barpL7hWi1gkpuYc2KyYlpXaU5jSx3aDD3tJAeWlo6y7TaUUf06uM/c6JL2bWadjXTOzm6z69Lq+QVsnNlGTajueVBPpPmj4xoqy'
    'Nad+Cq9QWQuPkLRr9mY1HpNvNl9MmySM0/Me97KmDm1TAz8jrBpTAMbzy7nF88orFb8s1flRvMI0wFDiUz6sVO4yU3kf1TvivQsV'
    '01DQkI0UXFT2kj4Z0E1x8tRYK0HuoUfRKNppXaxKE8qQsPfrltaz9FFks15mqlVNSeyMaqYi8gjcsyiIO5JNrpJV8N3CjTuUTIFq'
    'JW0KHpBE841VkbKQfqR28EE989FC9WHCSVFK/vYZB6R5LoznvoJsviG+ZpSYytpu9fA4mi5eu31zDBsf1jJDz1PKiqF9QDVvl74R'
    'bGgmeYRJ6EdjDUUWwYGCJ3ue6yAMejI2fFKvPVNQ6s7poLOn9a9K16+eim6XlZUkNrOiEWqxlZoNVyvYtM8pYFQ33wr7a7jCkY8l'
    'EYarfQfjbuhLbfUiaPljZtz109EWPzD+KNk5/V5Qsp2cv3eGL6FVW1wTugCi9OR7aHTm63/gzMOFWJXBmDyDoWUXTewJXhW6y43Q'
    'nzPNjmvudWi8tQQXzywit6LvKfFF0zWUhv2pThWejDbsuHbjW0OoZkfl+E1v30V8dkgf6Dn5Cjqsf/e8mRAdbasG1e/a+Y7nT+H3'
    'NL8FMF+MCFyM1nXdRITnsO/owgsNBW7wEnYiODUfqXg8+dwN9ed+T1iMmxzHELf0tQUVwFTxdyhn/Bu6Meku2TXlYcVXckStbXuq'
    'U/B1EQc53hTCzVR/i8SGaGB/o8gi8arQ10skS5V8KS8M5bGaDPE6s+BzV4o9pCrUVQ3fNqVLHPeBeWz8OtosDujpbz/3O2zGb43d'
    'sMxBzr4mjbDhApnvMPSr8TTEmSe2FVnFO+vrODe6eNtYX1Pf9Br4Ef2NUpHuCVU3/K0yCtk3n0B8SpKdxSNyatHfVnTGfxOSi4QB'
    'EoyVSEWFfb6zQzNvKRyEnE/tkjqXAlU/aWujgQ+tOTreWVOqbVDHkbjW8GwyuhpkoLVQbOtWZE4kkvXMTmdNd/wNQ80cGw8n/vcW'
    'hX1impr16ngQlLWmdU5eefeCQpPYoo9WNn9ludOkivHnm/n+BZJfQVBZ2R+eoayzqhSXrZMxtdCfEXnv1f6L3RY/GYFEk1h6JkS6'
    '56DIVsjOH4tshS1rtlrKCL735JCD85F4VyB0HnBcPE4/X0E7DhXUXW22ojMObwyOW8tU/mbXVkCJ3+SZazp1+wbv3LrJ/Gc+utqS'
    'Np6iqY/PyZcZea81UYcKsplFgBnd9tjJhM7R/vNmOB3nuzdF1ZAqXT7Bp5IwNnGKXFN7aE5N/kjbIDgTYVbSg1QYiNewlgoYXsNF'
    'wWHXxuB/AwzqLx4='
)
_SOURCES_B64['m1_lorma'] = (
    'eNrVfQtz28aS7l/Bkcu1gE3RkhMn53Cj3NAybWujlyXZSZbiRUEkKMHiAyZIy3JO9rfv193zBijJZ2vr1nUqNjGPnpmenp7unp6e'
    '/kYxLeeLZTSvzmfqZ3Vrf3+s5jPzcTmZX5iPaba8Mh/LYpqbj0U2G82nFtzVallMzGdZDK8ntvDXohwX9G1SZqtpeRtlVTQrKfVR'
    '1Jt9jj5niyrKFnk0n01uo/F8EU3mw2wSLfNqWcwu29HRLPo1u7yc5NEkzz7n0fIqn0bZZD7L2+ezV92zbvpq7yTawTjb+exzsZjP'
    '2pf5Mj7fON49SXX++UYrOkSV5Hx2cPSqt39XHVOgVun4j7vrHP/hVDl6f3ZXKyqbKpxvPLvmET67mS+uMehn5XCRTrfTyXwxzc43'
    'EsKWtHDYPeilb/cOzwD2fGOSX2aTzctPKBPRn0fA32SUL6JZNs2j8SK7nOYzUMCYkBYdX2VVvrkdHcxH+STaVmXt2Bi4hZsCbjql'
    'su3y9nyD+nB6cPRrLz3rnVLzr7NJlUfBn0fR2WKVIxdzdxstVrNoOY+GV/nwmrswBLgon40oFf+AMsbN2OGWMPJoh/qzfb7ROZ8R'
    'fK8H1BR3q9d7hc/vn5/PTt4fpmcn3b1Dk80pbvmT3ul7Hif3tOnPIyblaFVGmfS9nBdA43gxB+HNomy5zJA8ivJsMSmAbQzzfMat'
    'pkeH6aveh2bsGNwgax4tF1kxA9FHo/xz9GmVL4q8imIgaUaYwnoosE7mWB6TYpQti/msFU2yahnl5Xx4RZmrKh8lso5ezYcrnulF'
    'Xi7yCr+4BpbH0W76Ya/3G89qtlrOFaH43TJZMQ+RJgpNZpuAVUafi/wGo84mt1VRJUSsV3k2ErJdZlj+GzpNviKs4PON8WoyIZp5'
    '2+u+Snffvj/89RR92Aaaunv7XsLr9/v76UH3d5v491oHDWvwxqPb4Irpfo/mfHvr+fdNE7qcX+ezKioxW4Bzky1GUZlVVSsqZsPJ'
    'aoQ1J/Q5mVf0+7R3LFXOZ8dHR/t7h2+4xWmezRowyChUefPPaIJBXa1m17pdxglNHyG5sSHM5PvTXnp8cvQf60nzkVBNdgFmOIq+'
    '4P9yMf+YD4U8qmW2WFbEX6mFYgQyKJa3QHnv4Lh30j17f0JUv9XeeiFU0x1l5ZIYwP7RyUE3/dA92esqxgI8TYOBPrLJ8f4cFTbL'
    'ogPinVW6L4u5kB1jeZLd5gv0aTK/yReJIKCcrCpT/WmL1lZWJrp94pTbP+iv7v7x2y6nNKGBCYIWQHRxGxHYKK6wacjiKq+y6Fm0'
    'MIBfnRwdg9uqsd8LCtWoKy/yze+jdX8eRZkgj/akxYxmcpEtwVxo/lKuv72m/iNnzsLaNCtnNMNIOp+9e9872eudppi89PQMhIKl'
    'QfQoKZRL29F3TS2AuJbFZzCUKpuWE4yNpoSYzC3/qpZY2PEl+H9U5UvZfqcrsJVJtriUJRJVc2whii8BlW+7J6/Sw94bv20n/aR7'
    '+IboK/6uFX2/lbjduaLlNsOeIn0aznlzYk4DPlZNimGu96iXB89fgO3NNrlzkwIML66ui1Jo+jMNYDkvW9HNVYHuTrPb6CKPVrNJ'
    'dpHTMBf4+3M2W6LDveOj3bfETkDtv/X23rzFdtvb7f4hZIC07snB++P09Ul316yKNyfErfb3CNXbVEhYy8He7slR+rJ7tvu2xp4e'
    'yUKvs5ZotFowWzHz2fvQ3Q9gEb3Tst9929v99fhI9vXa4n8UXS6yUUEM3m5HANkCUvMRhg1KymilvHkZnX2P/bD7oZf2PmCOmG4I'
    'Cc9f0Gp4E6S+oL3z9HQPu9Zu9zh9e/T+hLnyVvuFx3SAetrKq4ymDzIgCGhegnnST+w32Hsq6gOVm+VflrIhEjHvzcbYymbDvBPl'
    'NHmYcKAEczW/oS1seDXHZqU2QTCqo+P0V7T/4/nsECs3fXOyR/t6HzIBSCKnnWYLOxA+h9Wkos8X/ue2yh5VUy7c3noeJrwIEraT'
    'AYj4j5doKt3vHrx81SUM9FGScp+/oL/x1zZ9P29vDQQjN3lxeSWCFTHvTSbbmDbBp7QLJmAPNC+QcgQnqwUk2mo4X2CRU1nexNIz'
    'yDKHjPDnf99SGb9uM+29UJ8vmTZ/fOFIXyxnKN5AM/s9S1FWMFKSkrcrPv+7JJpF8Vy+G1jM9271gPKVDFYnr+8kp05hqp0SS4FE'
    'O0d8K4hkOiInkrQB9IpsA7pCfzcZsWpxtRqkSOLbbSUb7x4dnvUOz9Ro7cg3IeVaPp8rvghyQ+PXeblkRhdsxwQPIuk0u85HxaKK'
    'lZzeivIv4Ebp/HqH1ieaFWWoXWEJxiSBImlWthtS9eB357NxcQn1JgNtUNdBgBujfByNi9koRf9S0pfixXy+xIhJhE/UXF4VYNKg'
    'yoF8Uq+pFIQX/rdSxXQe+p2WUOJa/IsAAR7Blt9UDSO8ySbX3FjiVKc/oCbWH1DMVgrK6F61s7KEIB8DHrXY/gjWFNv2eRCJVAXY'
    'ST6LqRaL9VsOyEW+XC1mrDnZEbcrKI3xdX67M2H5yClI2f2twTmzGYNCVlZStB5Tuyl4jRnbeL4Cr/JR+A1oshpaMStXkOJcnBEz'
    '2zHQ2izxxInNx8DBonjy26x1b2CHW/roFQHJcneUQT5ABXgHGKyYVe5BN6NVdZgvQyKd57apOihGiZ4/3Xt/rrjI/ZNFyKTKPCSq'
    '0gkGj9nJJjIkF0VBhxTcks0PTgKDNLMNgEbpxzKmXig4L98fvtqHCN09I2blL6p+bQYHrMAQkym+5qMUGne7vJ4wQ8mhujXC9Khc'
    '92I9HHTVra96S1NNk6KBMV+pYqekRgwkBwjXr9H/w/nyNWGht1jMF3FDewxRZjTqsn7qqPw/OurcfLXE8KO4Oxphd8bPRJioZlJU'
    'MroAnAlQAAbldkuh3xpQfPy7hpVgPQbGC0tl62DdPfzD+RpjRnRTgAydtRbgxBhMNpkk/aFzdxUgHrvpnD9yWIGaBs7GofuorsHY'
    'w6To2WBC9NQavBs7detNA3UUJFjnbE2LSG6BktaEDdpqHVwc/5E49rzlfDG88r/asxkb+Ga15PZ4NWOlB5IQSrw+n4kKwNlkR6za'
    'jq1FVXZSSK2lsqnDJMnC0fuwt0t6hwCCHAltgkTC1YisYsVYZdB3u6jS7DPkM1Kg4ySidU5ct1zxyEkQ7x6QDCRA28vbMhcDlEA7'
    'nyE7fXX2x7FtcIztfKnF+NPd7n6PyF+BOp9JoWk2W2FKmoQC6TCjWFolCvlnNC4hzpNVZgijAecqkIz/07PuyVl6tsdmLLLQtukv'
    '2nNccf60R5pNXcB/En33wxbrQGutbz4UssOxNWrpEmG9KcZF/ECILP3wcp1jB3LZH2FgcUGNYs7HmhEDvhiY2yRAxWNBRO84ZR2B'
    'kPkSy63Ky7SAdWqA7GMoc15mmY1M5oej3e7L9HTvP3tOgc9A90VagbFKIT1JVTkpSMoiftjXn2oL+WfEtTjTApUcsu4UI86Snkoy'
    'OqaTpY+yV+j+7qjCHV92plq0sRPQ6opUdtgiAQYb5ogNWJiibHKT3VbQi6trMWhgF79kQYzUL0oQsboYadH5zdE+oYiGRSo3BiVp'
    'pz1SRf/8C8uUjAE0Ytq+veF3DJ+SfF4qZLc35tq6yELQPhUjgkXN9LnmwCmuGu/HnNOiwskAXYGhInZq9JE+UObP366ypbFaQksF'
    'BmLGGnBQzZU1bo5lVinVcwjMXeTCXdiI6o0RQ2SlXbg1aab1oSqpqgkd9TEPwXk9AZ4wna5mpInIwJIHIkcgaaGNRLQaShJfFxgG'
    'wLhQIHjZ7rRBSvFQgcgWC3QPOg1+ZLextK3yyvl8gkzqAa8sfGJpERZUO4lPvI+rTvR4ZOzc/xQASPknt+7Yp8iOGj1ub4/xa1Tw'
    '7y38zr5Iafp6/FibixgK6J5heFYrnFhEj6Hpm4EqcqIeYzywAlDdFg2yTU3GSMFQpU1VgvOyL5S1TQwTnJOqW3QlsDMSmIQJkSzl'
    'KZlDJjzZLxX7c5OJpP78KxC3GuzsUcwWhYNiuJhvvt7+5UdlI0mYabgQhXVYu7g19KtJvsB6ZB0DXbIiO6eyQQLJm2zr0hSjdb/+'
    'fZb+lrHBDzo1fYQgeAgh1uUm9KnUIPrZ6UpAll4f61UbCqth0j9mvdqMugzlVnJHJiWcwwZT0Jk5febCE0cGZNnEVSXh9ARS2TAo'
    '0zkNkXxqzsl3DkckXw4RhIlTAWPa4N1TCUnyz6S4aHvHscWsKmFmJsrEua/oGkjLoVPDlqW1D0jnNK7YyHcgKKpHu6IHt02pKcls'
    'LNumdEpL4lxd4AzERclFTh0kElcAxUAJfEx/qQ7wNp8voB6hWSkYG1C83vYPWmxMSQvwb2ZGKre9T7/evOu+ni92s1WVTfYP2twI'
    'VhqzdshhRrhvKZ0oJVjYRxmcNvJMi4oMQymsEJqB0xrBpyjVqnW11N3ShIX+ILEbB9X5G5+uTlOii7YYEbFQPVtrYe2Gslfn04uc'
    'N3hLt247ejPAb4evaPVI2qiU0Qn8U1Wl3ohAQxzNhZcI5a1gyCXyQSWvKJnk43DctrA79CQR5lRrAYt+y1NxT7CzQIDVCkzY+Rs6'
    'Dj08OouI29yqsbT0UDrob/QUJumF10q/82KgOoDZVyqhFfBoKqy4FohaOBs4xJlfJxIliGvRoJoAEboAi0UPU9BCvqMHPwWCXiMu'
    'GO4ztyM0brhoQPPaUPKPIihicHeQTzvaG5Pad5NVWnwRycgqXfD9WOHEBaIRw8Gk4mymWJKhFc3ETjsEgA/N2BaJRVXg3AXb6cVK'
    'JDHUN0ZWJYLJEPJP0IQqzx9BtFRBVH118DprIrqGxaZXG9Dtg8HSRdrNMtdgvekOyq5H0ObPd67PDcWWRCkkoW7/QFpezmPR7jit'
    'TbiKldhZUjtIK7MF+DCOFCtjGIM1GegqsOuldACkvQqokUMY5U9JdcHhS1y24VmTA2K0DiDaeiU+Kkrb8bDRhjpR5v3tQaj+86on'
    'LRjlk3bKO2CaCoNg8CTVtZ+PD8BYrgoceeCMiCj78UikL9XLZzgM/QFbo/RAhCXqKRwpFkrSgJiVaut8ajBaiZAxM1kq4QKbz7XS'
    '0+ibalf5pxScRhJI/sZBVGp3ToIyXH6xsgqmgJ0xQEzUj9gs0Rb3y7jxKO5dgNZwwI6jrJhrkufAMhGphiH97NlIQysKdyeKH1dJ'
    'x2BHmuHaiW8uticYP0t+aClt4hS20mNeovCQumRfhGzmrD/dFd0JU6uhJxeLPLu2NvaLbHh9QbYtmeqYTVok9Wt6fRSdkiyF811k'
    'iCtA5i91dUBKEjUdIZakpOZjkvD0Goxkw5betqIFr0hHw4XiO53kVdWWJs+YnlU/DB1L3hdZe6PFvFT07nQ50aoZbP6YWimaomie'
    '8oE0yUhnLQe2GGpa0Zf2iFaEo7FdTK7VkmO6rHxhWCxAhSzhNGfnDSCAKcc/+g0mmVCOg9QY2jta2MH/ScMhjLKRI5cK6uEI9bb4'
    'KFn/Zt6RsB3dBfBFm7Icw1qMdjHSFrF4CN2QcnGstdwRCLa6Yyd3oVFfvnxTX9QgiG3P0nH8JTlX58cH0AfZdmaO1nmDyi7RKbH1'
    'Gsr+N5yh3Mx0OZCHZIv1cc7Yt4x1Mb/IjQlvCR8WrCB7eEf/EBa+W7fj09878TYcHf6egJSFNHYMh6cm0BcIlJdycGf3ozhcRNyV'
    'JLFbhuBj7Fa2ZLsjxduS6VcSpGgrXGw7AP39ktTYzW0+yrHAnQzTAaUMK5ijYjxuAomTVQsmaWcXFdUkVdndQvyZ+1xF2sLOCR3y'
    '5ABP4H6zJvW4/d2YRFTo+gxZmn/c/n6sGCaVbHEqtQMnCygzatxgv+1/yF7gtTqa57Kbs8jkMEJV4HxDn8KyrpWKE1W83+JimmB2'
    'ad90TpnNtrBPUtzufv10zNA2th32CZN9l3xr28O8mMT72BZ39zWile+WNREBNnVAFHrRsgMlW8FF+4ELXe2cD5ssHxVn2BBiVa+J'
    'l0gntEIxxRlXAZPHLnCxjwm3nV2z9tn+Qg1hJU1I+8yEjUrlVtjN+3vAnIhOLeIq0W0H7c5EIsUBpZvGarWXptEZ2jWszh8axQxk'
    'R2lfB6/JOlKHp3rlKPnrJlS1/VRV+n85oV5D1KsHTBu3kdRhfXRhsZ/M/X0GT6G+xh+BDfAv6f1WbdmQqwC2VbKlSlriOwxIomwr'
    'IvLsdrHzKhO3lnAgC6ZijRFLhJFrlEVE26TPNz5pR9chzXsxShz7cM42V6eVOndwMvtUQS16BR6nglBvYWACy6exoYC1IhTkx7Aj'
    'XXA5wqhSZnxxObWWWCQpsyulCdRB42K6EwZs5qM6BOX85jtWMBPw2CmZAEjqsvYxd+oFiCGfUdWvOhXm22G42p5cQxyaVvYzd7p1'
    'kp5Y8uZJL2gHiC/BUUrdOsvI1dLyCmMy52I+x6VBDMl4oWoFxGthSUEroOqcp+TvbBENNkmMPWbsSLcgJSX6UAgUQILmDoqBEf7w'
    'vQIoKo2q/zVfzCuvfmMdXsgtGVhOuiK5m/qYYLU56Lvqah9VO/uM6TZOI0kdDOA7BbmcmDUcDxnpc7+gzH2LBzVb6iSVDHR8RYRJ'
    'xdGZW/USCmLiadZ6tvk0QslXVy3duh4pzchVoCrQ6Zv2894xrtruup3fVEZgzISHaSBbAyMC6gPopLbir/oEwnSGXNMG+ihF6bYh'
    '/LM62PZqVmEx51+xsao26KyPWJMF85NuxClNEh5QdeUpL7Ad4LyDKl+BsRIgpwYLhWRd2A44qVR61tAIqviN0H/DCTnDHhuf5xiu'
    'AAdszrXsVfzHITPAHRr9KElpXa7xZ29F+vIEaYS6jFidygndqdDKJpEB+wT7dyK0bpXCmFQs0xSbxmQMVLsroVqV5KDUNmWcCaXi'
    '1qyE0RxrQ0ss0yeLkji4u7/eoY6s2W+bt1rbOi03XlF8fGMHppVsGdeXpL79fIl+cQG1WYeg/9h9HVaa9/s9c1yEtaFvJihIfiln'
    'arWFZ+2qhIFY7a/asZMQoXs40/yH87QitRix9Mw2QCU8OC6FJNoTDGeQXMNxHsymF6MswmUFCxucKEm0yb0qpnAJWZiluZxDDrii'
    '2wtkaNDGB7Xa6ETO+Ji67rHoCxvp2KkkdPVWK73I6QzabpacABIb5V/CLZTo2pICpOhZSxpPvA2b6vF4+1yjI/WeSlFH02de77Th'
    'kRvAhCfCVFxvyBZlHrdvOVtRbX9tpHvtzxILN0t5q3J8bfTuZbxs4HQr9pId7f0S9FMZHXfqtqlR5fRBcYIdf2fgf7Tea7cJb893'
    'Sb3uAlOD7xSPJbGpF6/BAXCxbwIrgipFSu10Z3PbLc3EouegBuyOyXOoyshU2kYwmaSmG8oxCqOXtkiuVutqBoWishaSfFoub+OZ'
    'nh9JJKnGL9/37ClONxoqDsLdLvDwtT3tK+jEVGyq3hEdH2A2G6cQT0V6D5hLTVKFNYMZi7vsrltGfrdyEgNLOg1+FWv0hWa5dhho'
    'RNy+Ob1LnF6SAVwR6xqG6YwgXYaGLE6uo9xIE6YNX1jhbcvAspKlDL9lu2Zkp/sgGvmCgbWZFFLyMKHzZ+m7A9aff3eNCBxnjfgz'
    'ziqKVs3oKECpOztswVdz8klkbt5FXiqV6LbRZaUYu0DWrHoFjv7pd5zihrH7ra31j/nE596hAvWJiItguy1KSU0wnioKLym9JZaN'
    'APlMqPQBlusAskpbaoD3yC3viEs4K091U5t423CudIWn46B4ub64IoRPvNWU/Pc7bBHHuTYOX8GTC9a+W7202dSnnL3oWhE7Zlzi'
    'yhl7F2l/jf/zAI+2WfqpaUbW0E2/83xra+AZpT65LloOl6jPmkYzrDhNc9ZMPA0tDu9tUU9r0uCa5dxhEy8sOtUjLy0kkOJPV5Bz'
    'SYwZrTjIEpusAs+uUzAcY+iJ+zV0VX52WqG7gnuHr41V+XxjE4e66uzU0dbZ8aeKZcpbHhkMMY3W4PUOoN4pfx9y3j22vj9S0MHT'
    'dOXte/13AD6gfU/6C750necl8Rnl/2HIPNi230GBm64atu3joOBxU0G6GfUOMvhx21jvob6l87GyRzk6e+nvRaVz3Gar9UvaTgvn'
    'dlBLKNZW/BRUBHa4hAAINq7T/qKlQX8aEGw9a/b+qyKUgo09dCRApy9Eqh3lbKq8AnLKhK7hLepTO9kg28ltSgiL4fRE/7bkbNlx'
    'naFUUczlHmBdo1Gm1DGpa1ZuKSpJiE+TBlhya7B2kee0Le6z5OY0if9LILTYSc6ZZ9K52aCKNSTdhY+vMQi4+n9KC4itQDcQYcrr'
    '+FqoAUTHJ67VfbQH9yS6Scf+h6d08sIQayhAseaeq7lrQgFfg3Qw8JXGD91elqaMK+yIligu02o+XpKB9qsMCObZp+tzt5J/tcdy'
    '2v2BcCVn3dR/h118ZVaRkn0lNqcK91ECe7PKhQ0ki6EjmIE2TBfTkszmO2LYOPWPftFllm7WEMyWoRd2ad1RFRqbAsq5Q6pvzLbE'
    'MqNa2WQgZP9WHa51B3FUqLyp++RJ9Dx5QGMyqfGpaeIZWHwG/fkT9GeaUVxk/8Hfkh8wb3puQPC0ddHSFt5tDrH1Arq262bbiGCs'
    '1556ywVTAAGSnLjdfntXIq2+DAB2LfpmvAYl+KPSo/qLAZpkoa12Nw2LV6tiJPN9HPjrQmeiXA1XzsZGWjmcEMc49gXyqKVKb23k'
    'VeygZ1lao/QsZf9m57sU6tj6Fo5/qXzNCRa4euJKDwz/qVhfLr0cbgk51w3qjx0EMNfvXA/qF0lFTWqwY2F8T3e0HZjGioRnJHjH'
    '0iZcrg0ih362YEpnj7f1jX7VJABzHd8hh4s9x/op8T9lg8xVSX/CEFwAabgIum3nDQmkeYIlnvqSSDBpjAvmOTWqv39W//dm0UyY'
    'nqt/dZ60vb5pLkQm38QfdScekRhwzM5+7CYoirkhr3znWJYEP4/0hcaFCmkAkz9ijUhkCn1lhX3dVeSpYVHewiM5IysFDMRV6bgp'
    'TXFvngDHDuqusgkJV7W798+e6Svq6oCH6tARbFiyLnKQRO5iBUIvXEUgVsJmhYnrs2pI7Q5afDiySb87g4HDHZkNp/AJWBRfYr5l'
    'YToMXlcu2TyhHQkU93P4F/uG+jyQ9E7MpMBy+syI3glwU8umf/r8+2/6EGdQOwKj/NqdYNtle3hPX/3N7UFy3xVmnuqWvSYDTOJ6'
    'BeRLbqulEJxKfigdKbToZhlW7ZjKKU9IMyYZdZ3GlGa557vniQ+/cVTgHYQNbi+44q66VEOSSk/BS9zju601p3a6t3eV1x1uPs8N'
    '23PpU3dyXVtuWcoJToFQoFpo0o111ZbbZsveHRK0ka8U78o7sbqGk7G/ueMtnQTrWGmB9i6YYr16mRynAuUB9g5dVE/lfefcpSbc'
    'd81tNBlp3j2sDec83rDw3+l81eUHur8kegLXsTEUkOEEhX83Eo7iBmOZuAuEtCI48e9aYKKTg5mYt3d+t7JWSPeG8AoNCmJ8DLUH'
    'pB6rZjepmacUE4UFRfOh/cUU8WSVzPvvLIBmuOG+Q8eJCwSQgZvy2oazz5cMgfaW0cTcy7KaF8Qr6KTp7Ks0g6O9HJsyk5lyK6Fe'
    'igWC3MQEB0x6agMeM+aECazpBU4FcDF5h8o+jXRQlicRY2IzUmFZnuofTzDovunXAFjhUWjrmbrCsENI7ZsZGaAawD+Re14E/ymP'
    'ErW5dan8Gzt/eAtNwWtgWa3IQMfaxRG5TiHuJQlm+SkiSKwBxaO8d82U966tNhyhjO0ylhTHqFE7p4/JzPFb+4xACUkk69nt/x82'
    'kHWGDPjMptUNFK1Gm5UISszTjI7KX/9J9nFHg1VFDL17RxbqLNR1VmF9j/3Q2IQqfrauhHe6pQ/o7jep+TKjY4yhBkxMphBT28EQ'
    'XJPOlm/TafAKmwh0PwZTgyRKm2vGdgtI+w353JXnpDdu1zMbvNuCOrSfA/wTNSt3lX3ua7+hhWE7CS0YnptQqCM813PA4a8aapBG'
    '0hIFxiqQDbrjuvo0J3pD+vN8g4mIbp3xD7YM8+RTkvzi2xJzCvXXUZNHd+cxfZTAP1rsI0Pn6pSEX61mfJk/AAAlCpcoKPxNh4aE'
    'O/6sVdEntzimKJ8Y5l8NQxDCdxQRGlJ4wg+P4nhz1BdIdDjVN4N1TvuXdKhD9hBWSHD2j/h4C8Qg4OsKY0R8WnoLnFrywgxRHIc7'
    '1tH4MlzlD1t/48u+nQgtE9y3sqSWTNVAf6qZGjhCqWToCRvU15DmZgq9dkmENZ/cwasUjG9aHcY5hDx30vH2Wt5psPkAZLr3FNaq'
    '5O5qUV1/wOqiE4z05OjMcx9Ne3/0Gv1JcfUG51y4VAIrEF2JANlrzyXH+Unyo5tFsYS8TUEaqF6HT03lQjtdwChIjpMglxyPL9eE'
    'ivmN6YTSQI5YAF43x1+U+Q0rIKXbBOI/BMGm1NbWmXUOoA9YeK5zvpPhuptxsZ8sGDaQ+ndjjLOR+JZBtjGlAxRB28NOME1jOcHG'
    'CuLAUwZFREk4cZBcOZHEpn4BrwBgL8O5gwJAUWIpqEx/kwEoOIMHHWQKcJaU0jgcvWSS+VXM8AwXOJN++g7ECIMKSvvCDmByGm7G'
    'IZ4+S47fqkPPkUlSR07t8EzfRjGEno8JofkjGikS8jWPRnribilj1OIbr6qFxLtlrC4vWjq9x5HR9HSn7mbonm0N55Nvquua/NVN'
    'EgaxqbiqzrMd7aP7Aw6mRhc/9JAVIQXFfKznt3kjxpXXYCSiLe6nFpOlv8QqoQwcLJkglogBcUmBir4Z44YROBhHzwKvCupniDYX'
    'U9nlvVj2wPepSosrGndAg1vTJY1bVPDR6pdwBFqiSsXpDyxKiWALCsBxQId/1VUxpkviclEPiC2Il5OdKr5ZYLsiJ82ML3Ek7eg3'
    'xeIY4eKsqZYEry99qY/4z0Gg3d5NtwftJspF7IAyo+B25Iqpj32YhIN1euD7rSjEHCiXXaozcJ1ZOX7xPnurNnmzcjbujiNURUEX'
    'qDnsLaKJ8xm6ojAVhq0qRib0LTjXV+CGAycDVVuEXUlr8bkcUr60dRsRhyrusIoYxzbmMeIkdbFt7yVUnP/gznJE9PuVtHivYNyN'
    'viYOQFiyBOBxDCgWQvBHAzyJcLJHwa4MQUg1IZuRSwu6kZdzzLu4KOZfsiEulmvP3fDKfyzD4/Cn/666Rm3tJQYFlMhBZPm6KRqE'
    'HVgP9GsUn4DN7CX/90xfP8edeWwiHHCZ4iGze6W49iD2p0RjRMyt9lr/4IuMlCvsTYzDlt626drsoshmy29yHyZg7LVY5b7Kxaqu'
    'KdJ0KdxoBesuhxs+ksoRGQPyLnq7V0C5LarNxYN00jUXQZoaLlmL5VeQX4vDHeSXLBBRg4JAp8MIhO93V1ZlAIAfRUi7zb7X4qlI'
    '3m8pT47lmr5DXIOh1wH+8k7HboG8+Hbg5Ia+TN05dd22a7lUx531CxXDlMS1Z/xKhxzSyhQ6jfkylYu0lpFYvsEt3UUM48CjZ33t'
    'yaMNG0N+nQu7gtfv4KBcbVrGt4R4GD+EAKYAsY+iPEzoWvlkNZ0Rn+Gtut1u3wm6CzJv7m29YEMfAJ6vaIciWkRbQxWx75gKA2Ej'
    '7T/IB/8rn+CpFW7k6vVo5KD5ARq/fkca2lf/ArCf6ekeSGq5y099GJUhvMlEh0EE5hcPUe1aU7hIyxZBqvGLTyhry8q/Txxu0RgG'
    'ltg4FyXF4qunWFj+32GBboStgeV+EE5LKAdepu4m8HB39K+K6SiHdO1/LiplMAsHeiY11/DR5ReWndKVqPz8V7ylbhphVlZ1TTYJ'
    'MPQVbb6y9zeI9IiLVKsLFTKJAjzTXcIlBeeh+B0UC/BGBVQy9+HVLUqnWFu8as83JFCgKpUz4yeYbuRfuanAQPodHI25tlldR4fZ'
    'kAQ2By61m3BDVkVHbF5X9QnzGT1RgGCMElBeXxpiqYlCXpFARYtSQjG04bcrAoaJKdKSPV+Fm0BIz1fd4zOEP0dIzTe9s1MnyJNg'
    'S5pXAR4oaaR6ZLdkPzyI5LZoAxEZUWI9CFL5uhLRYKyimFRtP5h00B1tt+NA2iaaUlDIO2hsjA2CgLkuoioVDxv/U1RIhapdE2Bd'
    'B9VVY+cI1X50WJZxtQTAYWK8R0R0dDOaKtWkiSAVdl4FPCXLWzTOb7hcUIZjOdH037FbcfSKVPyuPXf4fh/x+/9BQeXpYYpW9OOP'
    'OhrnYNAcu0GFJKnfKjFt2HAJQ3qISpaf7rTc/wgpiSgoGJbqOetJ7M/jKhagtktvJTMJaDyf6B/8Tor+UC+dBJNhl5kDjYKnCYGr'
    '5lWpYBCa/myhOychGy/5gOJ+xIGW1baZ+kEmFPI3Bda62BJGr5dXAeieeISSYjjIiB/ReXF1NV9NKMQMPHw7clHbadOGkPC78hO/'
    '3UJkKVQ+vCJtc+TpKUp/Uy3J61gadcfdk+6BO/1Td+4VYjs+uqWOufWut5DkIaVeJhQPSpVZHxLKB+Lg0r4pJOM1kZ2ghpHLtw23'
    'yZEzlOM3XLb/NK12Wn/J+xsSJdP25lkkMaDcg331dA7dDYdBWJMOfwVnf4oDU+/xmlzIOVohTj0VjMCpGJNQkQ1GsZtS/OtYXTXQ'
    '6xetGMmluUQtnsAyt0Pi2GT+uPhv+/zA/2gwUDfJTuuM6ZsEeDt4sW6ySOVixLVohZVeqkoXbqWXppKOnkLchQmJY0DGTiQ2CXa2'
    '7h0lvaC0FRxjhKfFfDy2iLtj+UhHud2mJuwzTRr8fOYLStS5NT3+hg7oCiKhmMvJOO+gdWS8V8RhnpioPOJyery/JxKHiTg84D3e'
    'e0tOHnPgMMTr7qu44DR3UFVMY3t4BsxlSuYCjFu5U4tDvObOSz0yQi0QceCAafvJPQmvHVGY56T5JRu//85Xv9PwJM4gYGw0Ecrf'
    'xoggDgwlfjgRvl188JzqJ66cmL70olWKF61MPF9DU/zeFt8Gg9cT5t+wNzNM8lJAho0EjYQQecYfgqRt9aIMQ47lMIocp72HwFom'
    'MrG53zOhe8r+81zG8nNV1PNsPAK6I0uxk7xoRIZeDDqIMBWV6LQ6dbjQglI80v5k3rkqdKyk/DKIgUSjMiBCklLFfQSZ0oKl+rtl'
    'rQCoxheHdQZPUNNRi0GuHIj4UqBxwdB3s1TclPpVLSXiXRX8fimuRKHPcDCNGt5TY3MDg/8peuGOUxLt6TeLu3bgEts81m37BgVx'
    '7NVDY0fgHb63GGmnYOrRva/MUKHwIqt2mKY4b5TvrAK8L4fgYFXFS8D43NQvO5KzXXOivhSvxQH7pF3g8IoG5AYS8dpauO9wvSlQ'
    'LU079fX64BuXhRfewfQvLI6MxpjstLifNiG/FmuIL+4NG0xuVFCdPFmcNRjcbKaJmcOhEAzykzV1vPGr2Mz6GrmeNT2pT50q6rEx'
    '/6qnqSQuStqpjGy+vX6HOqRA6aW2K1luTkcHSiqm1HaM2r+gHBlAyHfOeenSvcjJPrlMiO5dy0YUYD5SdcQtMhXZGulWY4tq+Wbe'
    'C4r8bj3XvGpiKr63nnJw00RsXdW8dXMPPT+McZnLLk1X3B08hKzAxqDqPx/U7w2ksgF4HIarMJNpIEaNYYpe8nFQP7ZwF4isKESe'
    '6G8PaiEXKgPGoe/aqmHPPXn4N5hf/ZPkWqIn387mzKj+2VyQwEycWAnwjKQHEb+UUtbzhdFN0m3C/B+JutjUePFPrj/QXVK6R0dG'
    '4Fi3tMnkr/sJhUMESywHqfTE9lgp2zqQDz9X4GepHPeum/W2VQxeN6EccoXPi5hkdF2+NVVTMTkehqcP3xtTw6/CYJ0KYVRjRHP7'
    '9fis8X0u83AfaOG6XMpLWUuWiF9CuLyvjtbmKPS/rncKJ6Dj3qu01z3Z/8OSrpIYe4hX9upU30Z6u4fS/EoqDZtbfL1tHlvgb36N'
    'kZNUAt71ObOBd8wtkdWCDKOiV8JEMVy6gUw/plpztq87PChwiVdXb7NOIIzPtAhd1Lvtt5lB1Q8lDcz+NS28z2uUaY++/jTIJgfA'
    'wDrAvoeAyl6DBvpfgeptu0YXD2JYNC9siLeaak7ZfadRJ/LDGrSxbKaqSW8Ga7DqIizoWuwDcLQGnOPCPAa2Qh5wRmPw1VMbxsKP'
    'hmA2WW2w12pf4Dp6p4cd6x/6rRvHXuhq9LVbcvaOHBBnH14nVIVKliO4k2xYpcRT5CXzMC5qreqz2iulxpkfxymQKQggSQrytumT'
    'sAUlgSNutClK1sRtjuAdu88AP3EhhmESpC4FRGCorCoGLSmrN8GQLImf7sDUR4QcJIlF2T+Vwyad8UUuT2UXW14R+yd/+QqRwyJD'
    'lUgAO96+FrhTy8JW71Vrn1tyg53yNTwj/1BKG0+FT3+T0E6gFjmdB70Os9sd921lbT+HSSedLEJR/ZJf1NQtCCdPBaZ7yq0q6zGw'
    '/6mzSPnE0PYPu1b7DYxep5xsHn7zI0fJu2zeafsEzICqxDQxwaUAfh77J4dkmh+q5KoSlvSZU9jjsIhNXLGkymU3nWKJukO57VGI'
    'X6R25If3ivnCzT9Aq/RIs7qV8lS8ARBxWxZSSSFYdeuJGbdh9ucbsjJAAFvMYOeV+U0t248cquBIMttSVE71me39pQHT+u+dvj/o'
    'eUGSxFvB7kqqZPDupdm/k6b3SSnUmi5wT6xaKf2gZz9DecDpncBZH+qLeLgj69FLdOJqirCF5k2cHfWooF4quESC14pr8dPN9tSw'
    'ffXV2wGW8I0vvFlBa/aXOY0qrCXLZl0VyW2opShGF2Or90agBlhJRxXDq+7Az22toBWCVEF+Vom99RsKaunILavItqm4lp3c4s39'
    'NbP8N4e4mvxUrviRIr47pSbZEmso98gmIY+hj9gbgaZfVaNtQXW9paz4ZgmqRwRl2dlc+R54PIveX0+Jdj25S1Hkn4ZgOo0SIzXD'
    'lNFxKCgsoOmgo+nFK1BHkSGJjnScQ2Dr2e9oyuAnO/RUdzQZmFTDi+y0NzblTmnHmXRHdJVlSXhiynYmDDwSvgzLaektePAi3CEk'
    'H8W4qaQ/4VosYpaYmkNiM2OaV2pLYYobYeH7nJJB4WUpeH5TLl0jI9OZG9WA0mbW6FiXzOwmK0KVCYRRJzYRk2o7nhQT7j5oeCpN'
    '6ZpTPxJnKKyFh0jaNHu1Go/JNpsvpk0cxml5h1tZU4a2qYEf2F31KQDj2eXc7Hnl5Ypdlsr8JFZh6mDI8SmsZSohSSi/j+Idsd6F'
    'gmnIaEhHCuKNeLEbDeim625UWQtB7qFH0cjaaV6sSBPykLD1y5aWs/RxZLNcZopVTbFojWim3DYJ3JOowaVL0XNoFm7coWQIVCpp'
    'kxuBvBfTWBS+3vQjtZ0Pypm3h9X7wpOilGdYZuwZ6ZkwnvoC8psTivq/v3e8ppcYytpmdffY4zteu32zcyIf2PKCnqcU3ErbgGrW'
    'Lh3Yw9BMcs8ioR+NJRRZBAcKHu95qt0x6Mvo8Em99ExBqRung8YewyflTdr7gIMOXiqnTREkRLbLykrik1rWCLHYcs2GG5Ks2ufk'
    'VayrPwvba7iJmY8lnpUrfQf9bmhLbfXCaPlNUm768egZfzD+6M0S+r2gmHk5P1uKB02rZ1wSsgAcIOVZUzrz9d8p9XAhWmXQJ09h'
    'aNlJE32CZ4VCssAL6EQvxzXXMzXeWoKLJxaRz6IfKH5V023Shv2pThUejzbLce3Gt4ZQzY7KjpbevovLDCF9oOXkAXR42v3Qewgh'
    'OtJWDarftPMc98/hs9jfApjvNwYmRmu6biLCU+h37JRc0LtbCJyOF7vgETVSznzyah21Zx9+X16JcpPjGOKaHk1SnkwVPyc949+Q'
    'jUl2yS7phg68nEWsbXuiU/BImIMcbwjhZqqfFLMuGtjfyL9IrCr0CFmhL8qQOWoWEoWWZGit8xJ86nKxu0SFuqjh66Zy985Xj41d'
    'R6vFAT395T/hAp3xW3037OIgY1+TRNhwD9w3GPrFeBhizBPdirTi7fVlnIvZvG2sL6kvbA/8ayGNXJGu+1ZX/OQo3eswLxk/Js7O'
    '7BGhMelfukNE/ybEFwkDxBgr4YoK+3z1lkbeUjgIVz7VS+qrFKj6WWsbDevQqqPj7TW5Wgd1DIlrFc8mpauBB1oNxdZuReZEIlm/'
    '2Oms6YafItaLY+Pu93u8SWGbmKZmPTseBKWtaZmTZ969wtLEtujtaRvAXcXVZAtcp0kUoyy5o4MYlmBUlveHZyjrtCq1ytbxmJrr'
    'z4is92r/xW6Ln4xAoklMPRMinDg12QrZ+X2RrbBl1VZLGcGzjQ45KAWKjkNdhtC5w3BxP/08gHYcKqib2mxBpx9eHxyzlin8zaat'
    'gBK/yTLXdOr2Dda5dYP512x0tSltPEVTb8jKA8u81xqvQwXZjCLAjK576DxowlcF5s1wOs7zdUXV8OKJvKSrYik3rRQGf+eYmuyR'
    'tkJwJsJLSXdSYSBes7SU2/CaVRQcdm0M/hvQqsk5'
)
_SOURCES_B64['m4_lora'] = (
    'eNrVfQt320ay5l/BKMdnAZuiHrGTGd4oO7RM29roZUnO41I8OBAJSoj4gAnSspLJ/vb7VVW/AUry7N6zZz0TmwC6q7urq6urqquq'
    '+xvFtJwvltG8upypn9W9/f17NZ+Zh+vJ/Mo8TLPljXlYFtPcPCyy2Wg+teBuVstiYh7LYng7sYX/KMpxQc/mzWw1Le+jrIpmJb39'
    'JurNPkefs0UVZYs8ms8m99F4vogm82E2iZZ5tSxm1+3oZBb9lF1fT/Jokmef82h5k0+jbDKf5e3L2ZvuRTd9c3AW7WGc7Xz2uVjM'
    'Z+3rfBlfbpzun6X6++VGKzpGleRydnTypnf4UB1ToFbp9LeH65z+5lQ5+XjxUCvqM1W43Ni65RFu3c0Xtxj0VjlcpNOX6WS+yC43'
    'EkKWNHDcPeql7w+OLwD1cmOSX2eTzetP2XSeX25E0TdA32SUL6JZNs2j8SK7nuYzEMCYcBad3mRVvrkTHc1H+SR6qcpqyBe/nfYs'
    '0BRA0+kN/tOQp1QrXd6XeXS3KJbLfBYV+P+yiobz2bi4bgs9Xc7Oj05+6qUXvXPq49tsUuVR8Oeb6GKxyvEV83sfLVazaDmPhjf5'
    '8Jb7OURLUT4b0Vv8A+oZN2OQWwJ6oj3q987lRudyRvC9HlBT3K1e7w0eX+5ezs4+HqcXZ92DY/OZ37jlz3rnH4966jFq+vMNk3u0'
    'KqNM+l7OC+B6vJiDOGdRtlxmeD2K8mwxKTAlGObljFtNT47TN72fm7FjcINP82i5yIDk+Swa5Z+jT6t8UeRVFANJM8IU1kyBtTTH'
    'EpoUo2xZzGetaJJVyygv58Mb+riq8lEia+3NfLhicljk5SKv8ItrYAmd7Kc/H/R+4dnPVss5TXhttOZTzEOkiUKT2SZgldHnIr/D'
    'qLPJfVVUCRH0TZ6NhLSXGVjEhn4nTxFW+eXGeDXB78vZ+173Tbr//uPxT+foww7Q1D049F68/Xh4mB51f7Uv/17roGEf3nh0G1wx'
    'PezRnO9s775smtDl/DafVVGJ2QKcu2wxisqsqlog9OFkNcK6FPqczCv6fd47lSqXs9OTk8OD43fc4jTPZg0YZBSqb/PPaIJB3axm'
    't7pdxglNHyG5sSHM5MfzXnp6dvK/1pPmN0I12RUY5ij6gv/Kxfz3fCjkUS2zBdYseDC1UIxABsXyHijvHZ32zroXH8+I6rfb26+E'
    'arqjrFwSlzg8OeumxMx2vlMP3cPT9128+HZXvXhzdnIKrmbqH1Lx3XzzZbTuzzdRJvCJtS9mNNhFtsT6oyGmXH9nTf1vnGGFtanj'
    'F4QEvLqcffjYOzvonacYX3p+AVyCemjK5A19Ja7+bVMLwP+y+Iw1V2XTcoLVTLRB6/Cef1VL0H58DT4aVflSdrHpCitvki2uhYqi'
    'ag5WrJYu5u999+xNetx757ftvD/rHr+jKYi/bUUvtxO3OzdEkTOwZ+nTcM5Mnhcjlno1KYa55vWvj3ZfgTPMNrlzkwI8Ia5ui1Km'
    '/TMNYDkvW9HdTYHuTrP76CqPVrNJdpXTMBf4+3M2W6LDvdOT/fe04jChv/QO3r3HrtXb7/4ms4x33bOjj6fp27Puvpn4d2e0oA8P'
    'CNU7VEhW39HB/tlJ+rp7sf++toK/kbVQX33RaLXglWfms/dz9zCARTRJK2P/fW//p9MT2R9r6+Ob6HqRjQrigZZjA2QLSM1HGDYo'
    'KQOs6N3r6OIltozuz7209zPmiOmGkLBLZH3yLnj7iraX8/MDMPb97mn6/uTjGTOu7fYrb10C9bTbVRlNH0QpENC8BH+hn2DJYM8V'
    '9YHKzfIvS9kziJgPZmNw+9kw70Q5TR4mHCjBXM3viMsPb+bg52qfwFo+OU1/QvvfX86OT86O0ndnB7T19bFtgiRyYsbbYNJ4HFaT'
    'ih5f+Y876vOomnLh9vZu+OJV8GInGYCIf3uNptLD7tHrN13CQB8l6evuK/obf+3Q8257eyAYucuL6xsRUIi/bTLZxrRPvKCNIgF7'
    'oHmBICA4WS0gGFbD+QKLnMoyn08vsN0fM8J3/76tPvy0w7T3Sj2+Ztr8/pUjoPBWrHgDzexLFjSs7KCECW/j2P27vDSLYleeG1jM'
    'S7d6QPlKTKmT17fypU5hqp0SS4GkH0fCKYhkOiJK0YYM9Mr2D7pCfzcZsWpxtRoELdo220rG3D85vugdX6jR2pFvRjvONpsrvghy'
    'Q+O3eblkRhfsWAQPUts0u81HxaKKlbjbivIv4Ebp/HaP1ieaFZ2iXWEJxiSk4dWsbDe81YPfZ3kTWkIG2qCugwA3Rvk4GhezUYr+'
    'paR2xIv5fIkRkyicqLm8IXEVVDmQR+o1lSJBlkurYvob+p2W0IVa/IsAAR7Blt9UDSO8yya33FjiVKc/oCaWw1HMVgrK6F61s7KE'
    'rBsDHrXY/h2sKbbt8yASqQqwk3wWUy2WfLcdkIt8uVrMWAGxI25X0L3i2/x+b8IihFOQPve3B5fMZgwKRdQv72M9IiX9t6G8YcYz'
    '2T+w/UxBAtQ1iBVzkb+y4ZDoIdOKyPImW4ooC+UDUHKCQezO0TIU6Pqc9K1eVMzK1bJRVbrcGPxfnzYIarpTG/70cd/pc77+Y8PQ'
    'Nh4lAjUha+bf7ZA/gzLVDdOHyjE1lmKrMGMcz1foor8CvgJdtflwcUd70Z6B1sZjvogT+53x6uqKGxBQlgEGWQa2m7MgDqACjAEM'
    'GN4qt28d9ZSEcBL59afl4j6oflcsb6I5Fly8Ft9eR5OEBOZxw7R5zVLhNrGleJwoNdV+F+I100d/8i+8WHr8D9hpAP+BIQGX7te9'
    'yNHh56L+87w3Y4+pQHMcPWifu3CRx9kL0Q9V5lmkKp1gvkGQ2URm0aWKoEMKbsl2J+cFgzT8CQCNtQdsiHqh4Lz+ePzmEHpR94K2'
    'V38bqDORAWultC0Wf+SjFKaWdnk74ZnJoY83wvToRPdiPRx01a2vestTM18aYLwTVrFTUiMGsi4MA2/R/+P58i1hobdYzBdxQ3sM'
    'UWY06rLRwTH2fO/o6PPVEsOP4u5oBHkSPxPZ9vW2SiWjK8CZAAUgVrdbCv3Wcubj37WoBSwoMFtZKlsH6+HhH8/XmLGi2FkQNRNW'
    'EmDINZxtMo36uGBYCjIjw/TWRwXsgU2YYDNhsJUGA6/VfHjcerAsssVmN36ht9QXDRsqc25QnQwjaSCUgrTCnC2qEQndKNWEB2rU'
    'wcLpb4lj013OF8Mb/6k9m7GRd1Z73R6vZqyxQ4xHibeXM9Ff+TPZkqu2Y0tTlZ03ZLagsqmzRZAFq/fzwT4pzQIIShBUYdJnVqOM'
    'mM9YfaDndlGl2WcoF2QgiZOIljzx+3LFIyctsntEArwAbWsOq6FdzvA5faPMpQJ3DKa/1Dro+X73sEcrQYG6nEmhaTZbYYaaJFrp'
    'MKNYWiXO8q9oXEIXJavbEEYh/qpAMv7PL7pnF+nFAZspyUrfpr+I0Fxd9LxHanldO30effvdNivwa62rPhSys7K1celSbb0pxkX8'
    'RIgsutvd2GE5hIHFFTXq7Lykxckhg95mxbJ7mrKCS8h8DX5f5WVawPo4wOdTWCK8j2U2Mh9/Ptnvvk7PD/6z5xT4DHRfpRV4rBTS'
    'k1SVk4JUBGKNff2odpN/RVyLP1qg8oWsd8WIP0lP5TU6pl9LH2Xb0P3dU4U7vuJHtUjKJKDVDdmbYGsGGOydIzZQknw+ucvuIWxn'
    'FVSv6OqeNvRr1iLIdkAvRCcsRlrve3dySCiiYZG9CIOSd+c9sqP8+ReWKVmyaMTEVLzhdwxjk++8VOjsxpjj6wIbQftUjAgWNdPn'
    'mq4Yrxrvx/ylRYWTAboCK1vs1Ojj/UCZt38hTUNbpWFiAQZixhpwUM2VtXWOZVYpu8kQmLvKhbuwkdwbI4bIFicR28isUh+qEl+b'
    '0FEf8xCc19M+CdPpakZqtAwseSJyBJKW30haq6Ek8TWiYQCMCwUymO1OG6QUDxWIbLFA96CQ40d2H0vb6ls5n0/wkXrAKwuPWFqE'
    'BdVO4hPvs6oTPRuZc4x/CQC8+Re37hhXyU4ePWvvQMyF9YB/b+N39kVK09OzZ9rWyVBA9wzDM7mS4vUMZiozUEVO1GOMByYsqtui'
    'QbapyRhvMFRpU5Xgb9kX+rRDDBOck6pbdCXRFoNJmBDpJCQlW96EJ/u1Yn/uayKpP/8KJK+Gc5QoZnPYUTFczDff7vzze2XgS5hp'
    'uBCFddhzD3uQoyb5CuuRNSx0yUrv/JataXi9yYZaTTHacNF/7CSnZc5YBr7sryF4CCHW5b7oU6lB9KPTlYAsvT7WqzYUVsOkf8x6'
    'tR/qQpdbyR2ZlHAOk0xBZ+b0mRpPHB22ySauKgmnJ5DKAEcfndMu+U7NOd+dwy/5LodEwsSpgLHL8e6phCT5Z1Jctb0j+WJWlTgj'
    'IcrE2b+oHXiXwyAEQ6xWRCCo07hiI9+BoKge7Yoe3Da9TUlmY+UqpZN6EudibfWAOCryZyAu6q+jOki8XAEUAyXwMf2VSH/lW9Vv'
    'AD9gzqOgSndZKMgX0KvQSakamyK8Og+PWmw3TAtwexdA+5B+vfvQPZr33s4X+9mqyiaHR23uFZYm7wUQ3Iwe0FL6VErgsPEyRG3S'
    'nBYVmUFT2Nw0x6dFhUdRyFUHtI3AKU1o6w8Su9NQnb/xsfw0JUJqi8kcK9s7WSislVw293x6lbNE4JhInHb07oHfDiPSqpW0USkT'
    'Kxiuqkq9EQmIWKALLxFSXeHYgugNlbyidAAVh+O2hd2hJ4lws1oL4BLbnnp8hq0IEq9WkcLO39H5+PHJRUTs6V6NpaWH0kF/oTVV'
    'y4XXSr/zaqA6gNlXRiArEdJUWPkukM1wEnaMQ+COmGZEIqRBNQEidAEWyyqmoIX8QA9+CCTDRlww3C23IzRu+PVAVdtQApMiKOKI'
    'D5BPOzoYk554l1Va3hFRymppcBha4XwRshTDwaTiJLJY0rECmomddggAMUmx8mJRFThlxP57tRLRDfXNkYKS2WQI+SeoTpXnxSJa'
    'riCqvjp4nTURXcNi06sN6PbBYOni3d0y12C96Q7KrkfQ5o8Prk89Hxf4pmw0k+weohBjVYT8gulZ0TWpSw2AwJHIJ+SOyEHj/G6+'
    'gkyEmVHHKZqs/L63iVzSEkshIQL3xux8U2jCWNLrrDSqXlzH26YPI2lnV1WciBhlDSEa0I++jbGJngMaegCdOI4HyGftl+PkPzx6'
    '8dgCAYEcAcU8H7GMqPri8ELd5ANtdaL5rZo+2ioYGwn9FsJZzmPR5vldm0g9VmpGSWSCd2W2wL4L/4fKoBdHX6D2Ar1L6bRaewlR'
    'I8c4QTwnVRUnxXHZhjddDojROoBo6434pSnt1p9YUFaZ93cGobmHmTZZPVA+aacs8aSp8HcGT1J8e3d8hH3hpsD5LA60iTE9E0zG'
    'qpdb8Nz4DqKQ9ECEY+opHKMWSrIEPaT6KDE1WK1EqJyZT+rFFYSNW6WX84kIalf5pxQbhbwgfQun5qmVlAjKcPnFyqaYAnauAi+g'
    'fsSGw7a4X8Z1T1FoAVYBhxmcu8dckzyBlolIsQzJJ93QasbdieJnVdIx2JFmuHbiH47Y49Yf5XtoJG9aGLbSM14d8Iq8lmO3mcM+'
    'dVd0J0ythp5cLfLs1p4oXWXD2ysypstUx2xDJy3Png2ek+yM00B8YGFQOxfplae8OUiDIn+HkowS+Zgkes02IpG3pLetaMFMxLFo'
    'gAdOIQZWbWnygulZ9cPQsXz7ImtvtJiXit6dLidaFccJF6ZWiqYomqfsPUMy8UXLgS2GuVb0pT2iFeFo6FeTW7XkmC4rX/kRi18h'
    'SzjN2RkLCGDK8f1UgkkmlMPrI4a1Bi3s4b9k/WEhvlJBPRyh3hb7vejfzDsSPkJxAXzRpkvHkBqjXYy0RTs0lCxoNdg0lnsCwTm0'
    'skckLjTqy5ev6osaBO0es3Qcf0kulbPLEfR/tpUaPyDeCbNrdMrufkwr/wMnhnczXQ7kIZ/F2jxn7FvGuphf5cZkC/9VHIvH1tOA'
    '/iEsfLtOYKO/9+IdeGX9PQEpC2nsGQ5PTaAv0AeuxcvAbotxuIi4K0litwzBx9itbMl2T4q35aNfSZBitmLbAdhrrmm/3dzhUzwL'
    '3PlgOqCMHwrmqBiPm0BiW7dgwj3d8D5/5j5XkT5L4RcdcjsDT+B+s+b8rP3tmDQM2HYYsjRPG7himFSyxW+pHXiEQXlV4wb7bf9D'
    '9gKv1dE8F0GBJV6HEaoClxvaZYR161ScIuPDFhfTBLNP+6bjEmO2hUMSwvcP6wejhrax7bCPp+y75E/fHubFJD7Etrh/qBGtfDGt'
    'SVAd64oBR6wqgVFFwUX7gUts7YgXmyz7tWTYEGJVr4mXSCe0PjjF8WYBE9c+cHGICbedXbP22d5GDWElTcjakAkblcqtsJuP94A5'
    'EZ1SxVWi2w7anYlkhrNp9x2bUbx3Gp2hHcvaeEIjqIHsGGnWwWuyhtXhqV45Rp11E6rafqEq/b+cUK8h6tUTpo3bSOqwfndhsVPf'
    '430GT6G+xr8DG+Bf0vvt2rIhvyZsq2Q7l3eJ7xsjL2VbEZFnv4udVx1paAkHsmAq1jexPBm5RlnA9BnE5cYn7bg+pHkvRolzHpCz'
    'jd1ppc4dnI99qqAWvQIPXyZYJ2BQBMunsaGANQIV5LWzJ11wOcKoUsc24kJuLe94pczs9E6gDhoX04MwcEYyqkNQnrq+GxEzAY+d'
    'kgWHpC5rD3WnXoAY8hlV/apTYb4dhqvPD2qIQ9PKXupOt36lJ5ZcD9Mr2gHia3CUUrfOMnK1tLzCHJFwMZ/j0iCGZHtStQLitbCk'
    'oBVQ9ZcXFL9gEQ02SYw9ZuxItyAlJfoQEBRAguYeioERfvdSARSVRtX/I1/MK69+Yx1eyC0ZWE66IvnG+5hgq0fQd9XVPqp2DhnT'
    'bZw+kzoYwHcKcjmxSjn+YNLnfkEfDy0e1Gypk3Oyr3JYGJOKozO36iUUxMTTrPVs8+mTkq9uWrp1PVKakZtAVaDTVh23sWdCL9x1'
    'O7+rjMCYCQ/TQLYHRgTUDgdJbcXf9AmE6Qz50Q700ZnSbUP4F3Ww7dWswmLO/8DGqtqgs11iTRbMD7oRpzRJeEDVjae8wHaA8y2q'
    'fAPGSoCcGiwUknVhJ+CkUmmroRFU8Ruh/w0n5Ll/agI0Yrh+HLFB3rJXiU2BzIDYDfSjFF/R5vgUdjDVEQCmjBgNywnFSGllk8iA'
    'Axj8GCetW6WwBRbLNMWmMRkD1e5KqFYl+aa1TRlnQqm4tQpiNKfa0BLL9MmiJA7u7q8PqCNr9tvmrda2TsuNVxQf19mBaSVbxvUl'
    'qW8/X6J/uoDarEPQ/zjWBlaaj4c9czyItaEjjRQkv5QztdrCs3ZVwr6v9lfthU6I0D2caf7D37QitRix9MwmXCU8OJZDEu0JhjNI'
    'ruF4OmfTq1EWFR0HNjhRkugTk6qYwgVoYZbmcg45AOZTWqwgXWV8UKuNTmCNQ7zry4++sJGOnYjCuBS10oucfA7sZskvQGKj/Eu4'
    'hRJdW1KAFD1rSeOJt2FTPR5vn2t0pN4LKepo+szrnTY8cgOY0AOAiusN2aLM4/YtZyuq7a+NdK/9l2LhZuygt+f4Vundy3hVIUJA'
    '7CV72tsp6KcyOu7VbVOjyumD4gR7/s7A/2i9124T3p7vknrd5akG3ykey8umXrwFB1hMEagJnikvSamd7m3uuKWZWPQc1IA9MHkO'
    'VRmZStsIJvBA1N1QjnAYvbRFcrVaVzMoFJW1kOTTcnkfz/T8yEuSavzyfc+e4nSjoeIg3O2CcATb076CTkzFvtU7ohOwwGbjFOKp'
    'SO8Bc6lJqrBmMGNxl91ty8jvVk5iYEmnwY9mjb7QLNcOA42I2zeHr4nTSzKAK2JdwzCdEaTL0JDFr+soN9KEacMXVnjbMrCsZCnD'
    'b9muGdnpMYhGvmBgbSaFlDyKyN9A+u6A9effXSMCx1kj/oyziqJVMzoKUOrOHlvw1Zx8Epmbd5HXSiW6b3RRKsYukDWrXoGjf/od'
    'p7hh7H5ra/2hPrGfQ6hAfSLiIthui1JSE4ynisIrTm+JZSNAPhMqfYDlOoCs0pYa4CNyywfiEs7KU93UJt42nGld4ek0KF6uL64I'
    '4RNvNSX//QFbxGmujcM38NyDte9eL2029SnnPoqBZEeca8THsjeZ9s/5n0/wYJyln5pmZA3d9Du729sDzyj1yXXJc7hEfdY0mmHF'
    'aZqzZuJpaHH4aIt6WpMGVzwn4Fa87uhUj7zy8IIUf0opkMvLmNGKgyyxySrw7CoHwzGGnrhPQ1flZyclCmw+OH5rrMqXG5s4k1dn'
    'p462zo5eVSxT3vLIYIhptAavDwD1Qfl3kbP2qfX1koIOnqYrb9/rfwDwAe170l/wpds8L4nPKPcdQ+bBtv0BCtx01bBtnwYFT5sK'
    'UhjnB8jgp21jvYf6ls7Hyh7l6OylvxeVznGbrdYvaTstnLC5llCsrfgpqAjscAkBEGxc5/1FS4P+NCDYetZssL4ilKLSuT/o9IVI'
    'taOci5VTB3ssQNfwFvW5nWyQ7eQ+JYTFcHKjf1tytux4PtFbUcwlaLmu0ShT6pjUNSu3FJW8iM+TBlgS4lwLWztvi7s0ubVN4v8t'
    'EFrsFOnMM+ncbFDFGpLuwqfbGARc/T+lBcRWoDuIMOVtfCvUAKLjE9fqMdqDJwiF/bK/6TmdvDDEGgpQrLnnau6aUMAx2w4G/qDx'
    'Q7eXpSnjCjuiJYrrtJqPl2Sg/UMGBPPsi/Vft5N/t8dy2v0z4UrOuqn/Drv4g1lFSvaV2JwqPEYJ7L0soTl4LYaOYAbaMF1MSzKb'
    '74lh49w/+kWXWbpZQzDbhl7YhXlPVWhsCijnDqm+MdsSy4xqZZOBkP1bdbjWHeROovKm7vPn0W7yhMZkUuNz08QWWHwG/fkT9Gea'
    'UWTd+M7fkp8wb3puQPC0ddHSFt5tDrH1Arq162bHiGCs1557ywVTAAGSnPbdfnvx21ZfBgC7Fn0zXoMS/LvSo/qLAZpkoa0WlojF'
    'q1Uxkvl+H/jrQn9EuRqunI2NtHL4kI5x7AvkUUuV3trIi9xBz7K0RulZyv7sznMp1LH9NRz/WsUWECxw9cSVHhj+C7G+XHtfuCV8'
    'uW1Qf+wggLl+53ZQD58WNanBjoXxvdjTdmAaK15skeAdS5twsTeIHPqfBVP683hHpx9RTQIw1/EdcrjYLtZPif/oM8hclfQnDJlQ'
    '8A5hzzt23vCCNE+wxHNfEgkmjXHBPKdG9Y/P6n/fLJoJ03P1786Tttc3zYXI5Jv4oxJ4IG0Mjtk5bsEkOTLpPJT/HMuS4OeRjmVd'
    'qPwrMPnni6mk0dEhShzboLLNDYvyHh7oGVkpYCCuSsdNaYokHwQ4dlB3k01IuKolCtna0vk01AEP1aEj2LBkXeQgidzFCoReuIpA'
    'rITNChPXZ9WQ2h20+HBkk353BgOHOzIbTuETsCi+xBxVYzoMXgdPS86bMjBvCt98yK69Pg8kvRMzKbCcPjOi9wLc1D7TP33+/Td9'
    'iDOoHYHR91o4uO2yPbynp/7mziB5LGCfp7plw6KASYTTQL7ktloKwal8D6UjhRbdLMOqHVM55QlpxiSjwqdMaZZ7vt1NfPiNo+I4'
    '25m0F+TjUF2qIUm9T8FL3OO77TWndrq3D5XXHW4+zw3bc+lTd3JdW25Z+hKcAqFAtdCkG+uqLbfNlo0VE7SRrxTvynuxCrvKOFzA'
    'cXZPgnWstEAb+6dYr14mp6lAeYK9QxfVU/nYOXepCfdDcxtNRpoPT2vDOY83LPxXOl91+YHuL4mewHVsDAVkOEHhX42Eo7jBWCbu'
    'CinqCE78qxaY6ORgJubtvV+trBXSvSG8QoOCGB9D7QGpx6rZTWrmBSVwYkHRPGh/MUU8WSXz/isLoBmSG+zRceIC2a7gpry24ezz'
    'NUOgvWU0MXF4VvOCeAWdNJ39Ic3gaC/HpsxkptxKqJdigSA3McEBk57agMeMOWECa3qBUwEEou9R2ReRziD1PGJMbEYqh9QL/eM5'
    'Bt03/RoAKzwKbT1TruZ7hNS+mZEBqgH8c4nrI/gveJSoza1L5V/Y+cNbaApeA8tqRQY61i6lGVBviHvJC7P8FBEk1oDiUd6HZsr7'
    '0FYbjlDGThnLG8eoUTunj8nM8Uv7gkAJSSTr2e3/HzaQdYYM+Mym1R0UrUablQhKzNOMjspP/0n2cUeDVUUMvXtHFuos1HVWYX2P'
    '/dDYhCp+tq6Ed76tD+geN6n5MqNjjKEGTAK5EFM7wRBck862b9Np8AqbCHQ/YVyDJEqba8Z2C0j7Dd+5K7ukN+7UPzZ4twV1aD8H'
    '+OdqVh4qu+trv6GFYScJLRiem1CoI+zqOeBcfQ01SCNpiQJjFcgG3XFdfZoTvSH9ebnBRERBg/yDLcM8+fRKfnG0BCaN3sjkUa4E'
    'TB+94B8t9pGhc3V6hV+tZnyZPwAAJQpBFJRDqUNDQk4H1qrokVscU9ZeDPOvhiEI4TuKCA0pPOGHR3G8OeoLJDqc6pvBOqf9SzrU'
    'IXsIKyQ4+0cyzwVyTnC4whjp6ZbeAqeWvKRalLfjgXU0vg5X+dPW3/i6bydCywSPrSypJVM10I9qpgaOUCof9IQN6mtIczOFXrsk'
    'wprPH+BVCsZXrQ7jHEKeO+l4Zy3vNNh8AjLdOIW1Krm7WlTXn7C6tLM4gmpwgoVwEdh3KNgBBK19khy3JvlukmQj3QbV6/B5qKQm'
    'oNCKgiS0CmtBsu9QMJqZuZhj3DTkiEXbdbP3RRnWQNspxQmIZxBEllLbUWf22J8eYLu5zTnawnUk42I/WDBs+vSjXowbkXiNQWox'
    'pe2UMoqgx4HHT9NYzqaxNjibmEER0QjOEuSrnDViu77CeT+wl+FEQQGgfM6UHqi/yQAUnMGTjigFOMtAaRyOXj6SYVUM7AwXOJN+'
    'ui5sh/Oz7iG7qDW5sN0Ddb9sAyWQlbNJeUMxtwz2NdJswQlL9QVFMI4/cvhScJLELscBvRY7hwSWr/VQu8poewcOGX5Lk9dX+a0R'
    'DHaXqXJ/r2cZyxRpikY029G6qESjR6boFbqXFjPVlBdnGPRIFaN/gi8k6SyCd7JM9iKD4uB7yYuS4AlynF4hbbrfG/FVCABQmv20'
    '2+zYJ24wCxmacX7AP76zRYMRwYJ+/aDPoGBu8fWwaSclNrfdlB2bDspxGMOZ4ikwkEOKsQSimCOwKQsQ+E4GH77PcLCUeOWwAXKh'
    'XKYuWbguh7WvVMclHGQP7+i1zMLq5s4WnzAQKknFc56wk70WHlm1dOz3quQTaOvumX9Bf7GAnBiPK5XRkpjeFt9cIYcYhsycMfnM'
    'yZn5lln5X+G46Uwvz2TqY+cBh0xn8qx0Q3/u2c6rlqPh0Rxi9y3tzl/84C/7wd+ZvoV06SwM9WA2lPDsk0D8010IdEBt5zBut0Gd'
    'rskon7BGSLX/6aHB1FO1mLQbasq/z53VXdvXKKUdF6ON6N5zLibsEuVVqyuV5IPSuFLsxJJySUhC2hkyxUw9dl2qqBGnWFu8iC43'
    '2iYvJ0rlfEZIMN0kl+KZyUD6HZgCXV1U19FhxfKC1R89/KrhU0UmRa+r2qJ+QfnjkWyMopLJAxpS6qet263PW3PeOpABPt9alVsj'
    'On6H8sw5wZEQsUfJFyg8jiKoKAkH5xinDAKjpK1zDPCnBX+CH41kEmf/VURvwd+aV57EtNLKgwVM9q42Xctw9q53wckcObk3bFq0'
    '/f8uUVC3zu/Pzu+585s67jyuSueBBqMescV333RPL5BNW9o8V/rw+U8Hp6c9KIwHx70u51jXyclk1gWNKjCXXo0UZu2OptyHndBu'
    'KdEiDi1bfbI2iZeqLLTDbIkYRayC06u2nxHXr8sXc+xFNbojIrCWZCqE7ruodiAGWNEKHqeHbrQGBwjrE3wO1vDfc3IM+kZuMRKq'
    'oczaQYueebsxIh0ZOpdMvdq1n3eeSqUexn+Ugk6wFe2bVOQ6v6WaQE4GLIvyp4NjROKdfDy+CGab0BT0rvNUTFugBiP2VQMytNuU'
    'vq1Cj1CGRg5TFKLFmRMsHDeZhIeK+kLToQ/3arFFPKUdsdBJvq9gxphVPLBL8X6eikei5yja7yMN/z8oNzzdL9GKvv9e5yUcDJqj'
    'mlWwft3f2rRhA4mHdC2TbIR6bsQz+tzJg/Tw7N0tMs4RsOcK31jO1x7LZ5JH5BHfSqL+5QtJ1G91F4nlvk5dygIl/EK1pUoFPdar'
    'yxZ6EOPZeMl2usexhMWl4lpSP9ZaYXpTYK0LsdaVVSZ/CpeMUFICpzNiH3RsUt3o3C/bQkBemzaS2u/KD3zfCrFjQj5aIAv7yEvn'
    'oHLUqIYoT5qd69PuWffIneqpO88Krx0f21LHxH4qGSR5QqHXCeVEUUXWp0XxYTiItPfk8GBNchNk3iOvR5thkIPHle8jvBb/NI12'
    'Wn/JfRmSGNB2ZiuSNCju2ZZiHhQeCZuIJht+CszfajOjzuMStZAHt0KEerIlgVNp9aDXa3RC76GMv7FyttXrFI1o2a25QC2gdulk'
    'm+fcSv6o+G+bbf7/aCjQdsmc4Yzoq+RzM3SxAbAg6aCDXdaTxiqvVZUrp8prU0Vbg4inMAVxRrrYySIlmX7W3Xik15E2FImQNR+P'
    'LdIeWDXSTW63qQl7oZIGL5eyWamZOremx1/RAV1BXRLlaZHY2A/oIp2j4guoJt+cjzd7LJbqXBCseEo3K7GI3OSTkvOmOClrRmQV'
    'yyh1Gw5ZxMdbbw2yj7ZlULec7n0cuxd9aO+MZglApyJ6YJRTq1wjt6CPtod3geHS23UfTSsSg9vuvvpuTUYRArd2/1VtNey+3Pn0'
    '6j6VyPFQSv7a1UgmQg/NSsZwW+nfcrCw+4oFqVslRSmM1mUpQbSWoWzScnsFFG9fHmSVRotT8FiBy0r467oR3mfQJMB2DWWqwPtn'
    'OmkYezgT6JzymBhSvbqnTubNJEp7xq3K8JapNGxmjXJeG9ljubJsrSA4HJmPUr0zBPvGg+T3pMWreCCCctKYrnKic2lDrDMxkZp9'
    'OKnVfP2kmq8Th4TXyUYhBYNyBRtGJHIWwKYFtU40UrKJAsJ4qiE9tpLRj1o2kgqSTFBV/uERPacbwkW8BCuHtfns+AwvU/nn7gry'
    'sixmouY07aWWChQBiZ7PRMmkg8O0nEJdMnux2fIG+v31TZCDkAx+CvXCztyNy0lJ54tKJiOdMvlKaDBNAq55qp7CkMr5Xbyr0wvZ'
    'qm2qKDnBLmecIuu/m1Vdu/6iZriGyq0cFLgKqLGvjwK9JhOe8pznsooySRbVK8AdobBJ941hT23mk9eWms2kTukowtMy/7ztRJIl'
    'B7bi7xIdx/hZslg4wHH8NkV//nqMY67t0Q//Jsu8JjvJvJkwNd1Y9vhVFFjbj2V1mPwAPOaOCVNSMSu0acilb+enhwcXyoKlkrwP'
    'eOl717PKZU6c+X1dyJgLTusmqopp7AA3a7oakYlBcyt3aqnf14Sd1ZOT1HK/Bz7Qtp/ckzDyjzLrJ8033/n9d576nYYr9AaBXkUT'
    'oVzeTBpgB4ZKGOxcquDig+dUX4nppFGnGzBT3IBpUqgbyZbv5+SATDgeYv6NemWGSY5C+GCT7+NFiDzjkkQGYHUDHUOO5TyYYhe8'
    'i0NbJhm8CbGbUKoA/zpP4653U9S/2ZQgFKZOMo+XEMzQi0EHEaaiEv2uTh0utKAUj7Q/mXduCp2uLL8O0pDRqAyIkKRUcR9BprRg'
    'qX7PaSsAargjZdLHklbTUbv2QfnwcVyu8YLS4ZEqdVE9WlLZkm4KvjYcUYnoM5hZ1HD/KuFTwP8QvXLHKS/9I5qhHbhcJxHrtv0c'
    'AuJbr4fGvvh7HDocab986tGj15pRoTCWXMcs0EkZfXdWAe6j5Q2Wl4Bxe6vHG5O/a/NLnZdCmyPsFbiBzzkakCBA4rW1GxbC9aZA'
    'tTTt1Nfrk4OeCy/DiulfWBwfGq/BoMX9ogn5tXRfHDsbBvDrZF8q1bPFWYOjmf1o0lZxNhKD/GRNHW/8Kh2+zuSgZ01P6gunirqc'
    '1I+2NpXES1D7ddLBeK/foQ4pUHqp7csn90tH5yorptR2jNr/RDk68CP3VefyaDeWmt3imRDdcOdGFGA+UuWLIgoFNvqKAotbVMs/'
    'Dr+iyzas86hXTY7UH62nfEw1EVtvUW/dPELPT2NcJt6sKcuEg4eQFdg0cP3dQT10J5UNwOMwXIWZTAMxagxTAqHfB3UHDneByIpC'
    '8pf+zqCW9aQyYBz6rq0adp4leqvNr/5JxjWiJz99hDOj+mdzQQIzcdKVwDmZLlD+UkpZzx1NN0kBvfk/EhVb2Bh7KxFIFM5Noazl'
    'ZFXFuqVNJn/dT6hpIlhiOUil57bHSv/RubT4hhj/k/rihptah3fF4HUTRn+i/4mYZAztHLhYs3BzShrPGP9oWhu/CoN1KoSJxZFQ'
    '8afTi8bbEc1Fv6CF23Ip9xQuWSJ+DeHysTpaB6bbVnS9c/jh0dkXDr4Of7OkqyTGHlIGvjnXCt77A5TmW9Vp2Nzi2x1zvw0/8+3N'
    '/Eq9wFVqFzb3lQnUWi3orF40cByPDJduLuHfjX3GXqjzpNxBXl29zTq5aFiHc1Hvtq/1uXBVGpiiX35eY8736OtPg2zywQ2sTOz+'
    'y4fxHQf6X4Hx33aNYn9inGZe2SyLNYMGfe47jTrJV9agjWUzVU25BqzBqouwoGuxD8DRGuDRhqM5sBVyQjUaQ6hd6tQvfkISs8lq'
    '+7JW+wLv7QedXFn/0NeLOWeVvnkmCFS1YapA3NnHY9HD1K3KvpLlCO4kG1Yp8RS+HbyWmrhWdat2q7mJp5kvySGaAJKkIHehPw9b'
    'UBI4UrebomSu2+Ek+jEuSzn6eJq+PevusxHRQAwzlUhdMn4wVFYVg5bUNUoEQz7JFQYOTO2YxXnKWJT9U/lMUzKIyOWp7OXOK+Lw'
    '7C9fIXJYZKgSCWDH4d4Cd2pZ2PwSDeikTvBEn3IkrJF/6E0bhpXpL5JdDdQiLoyg12F2v/dL7+Dde3DN3n73N31Qj4OldLIIRfVr'
    'tqnrFoSTpwLTvQZLVdZjYBdwZ5Gym5btH3at9jsYZc75tblr00/eJldhem5xEzADqhLTxATGNnoFHcySTPM1wVxVMgNvOYU9Dov0'
    '4GyhlLKbTrFEhTHveBTiF6l5o223dzjm7R+gVUS+6cCwF+JwiKT3spBKyoKsW0/MuA2zv9yQlQEC2GYGO6/Mb2rZPuRQBUfysS1F'
    'cwnNILb3lwZM6793/vGo5+UpE4dIuyupksGtw2b/Tppuh6Zsh7rAI+mipfSTLl0O5QGndwJnvZ2VeLgj69Hln+ITjsyh5hqyPXWP'
    'q14qiOPC6WLtCgOzPTVsX311fYclfBOOYlbQmv1lTqMKa8myWVdFvjbUUhSji/G5+0agBlhJRxWDny/wc18raIUgVZBvsuOAmYaC'
    'WjpyyyqybSquZSe3eHN/zSz/zSGupmynN3wvHIcvqkm2xBrKPbJJ0J2AlHWOwgVo+lU12hZU11vKj8AsQXVvqyw7+1WeBx7PooOY'
    'lGjXk7sURf5pCKbTKDGyC2TJF7ZZCgoLaDroaHrxCtRRZEiiIx3nLPR69juaMvjWHD3VHU0G5q3hRXbaG5typ7TjTLojusqyJDwx'
    'ZTsTBh6Jo/fltPQWPHgRwngndFdAU0l/wrVYxCzRnoabGdO8UlsKU/gLhFciywdy6qP7K5q+UiSnOtYyB0X0bmaNjnXJzG6yIlSZ'
    'XDR1YhMxqbbjSTHh7oOG2ymVrjn1k+GGwlroyKJNszer8Zhss/li2sRhnJb3uJU1ZWibGvh3K6g+BWA8u5z7eV55X8UuS2V+EKsw'
    'dTDk+JRZNpWsQPS9j+Idsd6FgmnIaEhHClL+eOlTDeimiFOqrIUg99CjaGTtNC9WpAl5SNj6dUvLWfpcs1kuM8WqpnTQRjRTsS0E'
    '7nkU+NHLJlfJLPhm4cYdSoZApRLveLahKNxx6EdqOx+UM9e9qyvdJ0UpNyHNOPLCM2G88AXkd2d08cbhwemaXmIoa5vV3eOwkXjt'
    '9s3BGuw1wQt6nlJ+OW0Dqlm7dG4dQzPJI4uEfjSWUGQRHCh4vMecJfMpudbhk3rpmYJSN04HjT2DO+y7tPczDjp4qZw3JXER2S4r'
    'K0kRbFkjxGLLNRuClFm1zyn0SlffCttrCIbOx5JSzpW+g343tKW2emG0fA00N/1stMUPjD+6Noh+LyhtZc43ReMO6WqLS0IWQNSJ'
    '3CRNZ77+1dAeLkSrDPrkKQwtO2miT/CsUFYkOHqd6eW4JkJa460luHhuEbkVfUen7k0B3Q37U50qPB5tluPajW8NoZodleORvH0X'
    'kY4hfaDl5Al0eN79ufcUQnSkrRpUv2lIIWcX6cXBUQ/a+Hnv/PwAR/f73VMyCX4NYA6UCEyM1nTdRITn0O8oKIe6AjN4SZ6K8LYZ'
    'qZAGufeT/XfaUXdJq1ii8KDc5DiGuKV7y5QbNV2KLTfOsQWHZJfsmm40wJ2TIta2PdEpuKfPQY43hHAz1bf6WUdR7G8cO8RWFboH'
    'UPK9yh3joUOxlWRorfMSfOFysYdEhbqo4eumFA59H6jHxq6j1eKAnv7yb1GCzvi1vht2cZCxr0kibEjF4BsM/WI8DDHmiW5FWvHO'
    '+jJObgTeNtaX1DkTBn5sbCNXpIj76oYvbabgV3N5/DPi7MwekZ2W/m1FZ3JNKvFFwgAxxkq4osI+R7/TyFsKB+HKp3pJfZUCVT9q'
    'baNhHVp1dLyz5qvWQR1D4lrFs0npauCBVkOxtVuROZFI1i92Omu649vf9eLYePgKLW9S2CamqVnPjgdBaWta5uSZdyNxm9gWsOzc'
    'oaBS27IFrtMkitEniWRGGlkwKsv7wzOUdVqVWmXreEzN9WdE1nu1/2K3xU9GINEkpp4JkQJ6FdkK2fl9ka2wZdVWSxnBzakOOSgF'
    'io5DXYbQecBw8Tj9PIF2HCqom9psQacfXh8cs5Yp/NWmrYASv8oy13Tq9hXWuXWD+fdsdLUpbTxFU5F1ckU977XGFVdBNqMIMKPr'
    'Hjt3CtGOAXfFRjius3dRNVw6JHeRq3TmTStFEj48NKYme6StEJyJ8FLSnVQYiNcsLRW2tGYVBYddG4P/AqX+eFE='
)
_SOURCES_B64['m4_lorma'] = (
    'eNrVfQt320aS7l/BKMdnAZuiJMdOZrhRNrRM27rRy5LsJEtxcSASlGCRIE1QluRM7m/fr6r6DVCS594991zPxCaA7uru6urqenV1'
    'f62YzmeLZTSrzkr1s7qzvz9Vs9I8XExm5+Zhmi0vzcOymObmYZGVo9nUgru8XhYT8zgvhlcTW/hrMR8X9GzelNfT+V2UVVE5p7ff'
    'Rb3yS/QlW1RRtsijWTm5i8azRTSZDbNJtMyrZVFetKPDMvo1u7iY5NEkz77k0fIyn0bZZFbm7bPydfe0m77ePY62Mc52Xn4pFrOy'
    'fZEv47O1o53jVH8/W2tFB6iSnJX7h697e/fVMQVqlY7+uL/O0R9OlcMPp/e1oj5ThbO1jSse4cbNbHGFQW/Mh4t0+iKdzBbT7Gwt'
    'IWxJCwfd/V76bvfgFGDP1ib5RTZZv/icTWf52VoUfQf8TUb5IiqzaR6NF9nFNC9BAWNCWnR0mVX5+la0Pxvlk+iFKqshn/5x1LNA'
    'UwBNp5f4T0OeUq10eTfPo5tFsVzmZVTg/8sqGs7KcXHRFoI6K0/2D3/tpae9E+rjm2xS5VHw57vodHGd4ysm+C5aXJfRchYNL/Ph'
    'FfdziJaivBzRW/wD8hk3o5BbAnqiber31tla56wk+F4PqCnuVq/3Go8vnp+Vxx8O0tPj7u6B+cxv3PLHvZMP+z31GDX9+Y7pPbqe'
    'R5n0fT4rgOvxYgbqLKNsuczwehTl2WJSYEowzLOSW00PD9LXvY/N2DG4wadZtFxkQPKsjEb5l+jzdb4o8iqKgaSSMIVFU2AxzbCG'
    'JsUoWxazshVNsmoZ5fPZ8JI+Xlf5KJHF9no2vGZyWOTzRV7hF9fAGjrcST/u9n7j2c+ulzOa8NpozaeYh0gThSazdcCaR1+K/Aaj'
    'ziZ3VVElRNGXeTYS2l5m4BFr+p08RVjmZ2vj6wl+n5Xvet3X6c67Dwe/nqAPW0BTd3fPe/Hmw95eut/93b78e62Dhn9449FtcMV0'
    'r0dzvrX5/EXThC5nV3lZRXPMFuDcZItRNM+qqgVCH06uR1iYQp+TWUW/T3pHUuWsPDo83Ns9eMstTvOsbMAgo1B9m31BEwzq8rq8'
    '0u0yTmj6CMmNDWEmP5z00qPjw/+1mjS/E6rJzsExR9Et/psvZp/yoZBHtcwWWLNgwtRCMQIZFMs7oLy3f9Q77p5+OCaq32xvvhSq'
    '6Y6y+ZK4xN7h8X43/dg93u0q7gM8TYOBfmdfx3szVFifFx0Qb1npvixmQnaM5Ul2ly/Qp8nsJl8kgoD55Loy1Z+1aG1l80S3T+x0'
    '6wf91N07etflN01oYIKgBRCd30UENoor7CyyuOaXWbQRLQzg18eHR2DJauwPgkI16srLfP1FtOrPd1EmyKONa1HSTC6yJZgLzV/K'
    '9bdW1P/OmbOwNs3KKc0wXp2V7z/0jnd7JykmLz05BaFgaRA9yhv6SnvW900tgLiWxRcwlCqbzicYG00JMZk7/lUtsbDjC2wSUZUv'
    'ZY+eXoOtTLLFhSyRqJphn1F8Cah81z1+nR703vptO++Puwdvib7i71vRi83E7c4lLbcSe4/0aTjjHYw5DfhYNSmGud7IXu0/fwm2'
    'V65z5yYFGF5cXRVzoekvNIDlbN6Kbi4LdHea3UXneXRdTrLznIa5wN9fsnKJDveODnfeETsBtf/W2337Dntyb6f7h5AB3nWP9z8c'
    'pW+OuztmVbw9Jm61t0uo3qJCwlr2d3eOD9NX3dOddzX29J0s9DpriUbXC2YrZj57H7t7ASyid1r2O+96O78eHcrmX1v830UXi2xU'
    'EIO32xFAtoDUfIRhg5IyWilvX0WnL7Afdj/20t5HzBHTDSHh+UtaDW+Dty9p7zw52cWutdM9St8dfjhmrrzZfukxHaCetvIqo+mD'
    'oAgCms3BPOkn9hvsPRX1gcqV+e1SNkQi5t1yjK2sHOadKKfJw4QDJZir2Q1tYcPLGTYrtQmCUR0epb+i/R/PygOs3PTt8S7t633I'
    'BCCJnHaaTexAeBxWk4oeX/qPW+rzqJpy4fbm8/DFy+DFVjIAEf/xCk2le939V6+7hIE+StLX5y/pb/y1Rc/P25sDwchNXlxcivRF'
    'zHudyTamTfAZ7YIJ2APNC6Qcwcn1AmJvNZwtsMipLG9i6SlkmQNG+PO/b6oPv24x7b1Uj6+YNn986UhfLGco3kAz+4KlKCsYKUnJ'
    '2xWf/11emkXxXJ4bWMwLt3pA+UoGq5PX9/KlTmGqnTmWAol2jvhWEMl0RE4kaQPoFdkGdIX+rjNi1eJqNUiRxLfbSoDeOTw47R2c'
    'qtHaka9HWw6fzxVfBLmh8at8vmRGF2zHBA8i6TS7ykfFooqVMN+K8ltwo3R2tU3rE82KxtSusARjkkDxqpy3G97qwe+wMA0dKANt'
    'UNdBgGujfByNi3KUon8pKVXxYjZbYsQk5ydqLi9JFgdVDuSRek2lSErn0qqY/oZ+p3Noei3+RYAAj2DLb6qGEd5kkytuLHGq0x9Q'
    'EysZKGYrBWV0r9rZfA5BPgY8arH9Cawptu3zIBKpCrCTvIypFov1mw7IRb68XpSsXtkRtytolvFVfrc9YfnIKUif+5uDM2YzBoWi'
    'x8zvYj0ipdq0oZpixjPZP7D9TEEC1DXIJzMRALLhkOgh01rW8jJbipwOzQpQcoJB7M5RoRTo+pz0rdZXlPPrZaMieLY2+L8+bZBC'
    'dafW/OnjvtPnfPXHhqGtPUgEakJWzL/bIX8GZaobpg+VY2osxVZhxjieXaOL/gr4BnTV5sPFHe1F2wZamwXWOLHfGa+uIrwGAWUZ'
    'YJDlW7s5C+IAKsAYwIDhXef2raN7k+BN+oz+tFzcBdVviuVlNMOCi1fi2+tokpA2MG6YNq9ZKtwmthSPE6WD2+9CvGb66E9+y4ul'
    'x/+AnQbw7xkScOl+3Y4cA8VMbBs8783YYyrQHEcP2ucuXORh9kL0Q5V5FqlKJ5hvEGQ2kVl0qSLokII7Z6ua84JBGv4EgMaWBTZE'
    'vVBwXn04eL0Hpa97Sturvw3UmciAVW7aFouv+SiFIak9v5rwzOQwNjTC9OhE92I1HHTVra96y1MzWxpgvBNWsVNSIwayLtTBN+j/'
    'wWz5hrDQWyxmi7ihPYYoMxp12aLiWLJ+dAwQs+slhh/F3dEI8iR+JrLt622VSkbngDMBCkCsbrcU+q1d0Me/ay8MWFBgk7NUtgrW'
    '/cM/mK2w0UWxsyBq9rkkwJBrFVxnGvVxwbAUZEaG6a2PClg7mzDBRtBgKw0GXqt5/7j1YFlki81u/Exvqc8aNlTm3KA6GUbSQCgF'
    'aYU524sjErpRqgkP1KiDhaM/EsdivZwthpf+U7ss2YRd1l63x9cla+wQ41HizVkp+it/Jkt51XYMhaqy84ZsMlQ2dbYIMs/1Pu7u'
    'kNIsgKAEQRUmfeZ6lBHzGasP9NwuqjT7AuWCrD9xEtGSJ34/v+aRkxbZ3ScBXoC2NYfV0M5KfE5fK1uwwB2D6S+1Dnqy093r0UpQ'
    'oM5KKTTNymvMUJNEKx1mFEurxFn+GY3n0EXJpDiExYu/KpCM/5PT7vFperrLNljyQbTpLyI0Vxc96ZFaXtdOn0bf/7DJCvxK07EP'
    'hYzIbEpdulRbb4pxET8SIovudjd2WA5hYHFOjTo7L2lx4kLR26yYrY9SVnAJma/A76t8nhYwrQ7w+QiWCO/jPBuZjx8Pd7qv0pPd'
    '/+w5Bb4A3edpBR4rhfQkVfNJQSoCsca+flS7yT8jrsUfLVD5QqbJYsSfpKfyGh3Tr6WPsm3o/m6rwh1f8aNaJGUS0OqS7E0wpAMM'
    '9s4RW19JPp/cZHcQtrPqSqxx2NAvWIsg2wG9EJ2wGGm97+3hHqGIhkX2IgxK3p30yI7y519YpmTJohETU/GG3zGMTb7zUiHPlPE1'
    '1AU2gva5GBEsaqbPNV0xXjXej/lLiwonA3QFVrbYqdHH+4Gy3f9GmoY2ucPEAgzEjDXgoJopU/IMy6xSdpMhMHeeC3dhD4A3RgyR'
    'LU4itpFZpT5UJb42oaM+5iE4r6d9EqbT65LUaBlY8kjkCCQtv5G0VkNJ4mtEwwAYFwpkMNudNkgpHioQ2WKB7kEhx4/sLpa21bf5'
    'bDbBR+oBryw8YmkRFlQ7iU+8T6pO9GRknDT/FAB4809u3TGukhMgetLegpgL6wH/3sTv7FZK09OTJ9rWyVBA9wzDM7mS4vUEZioz'
    'UEVO1GOMByYsqtuiQbapyRhvMFRpU5Xgb9ktfdoihgnOSdUtuhIYyQlMwoRIbp6UbHkTnuxXiv25r4mk/vwrkLwanERRzOaw/WK4'
    'mK2/2frlR2XgS5hpuBCFdVinjvVSqUk+x3pkDQtdstI7v2VrGl6vs6FWU4w2XPQfclO1jANp4Mv+GoKHEGJd7os+lRpEPztdCcjS'
    '62O9akNhNUz6x6xX+6EudLmV3JFJCcdTZgo6M6cdhjxx5P2QTVxVEk5PIJUBjj46rjz5Ts053x3PnnwXD5gwcSpg7HK8eyohSf6Z'
    'FOdtL+CgKKs5fCREmYhsELUD73IYhGCI1YoIBHUaV2zkOxAU1aNd0YPbprcpyWysXKUUh0DiXKytHhBHRf4MxEX9dVQHiZfXAMVA'
    'CXxMfyXSX/lW9RvAD5jzKKjSXRYK8gX0KnRSqsamCK/Ovf0W2w3TAtzeBdDeo19v33f3Z703s8VOdl1lk739NvcKS5P3AghuRg9o'
    'KX0qJXDYeBmiNmlOi4rMoClsbprj06LCoyjkqgPaRuCUJrT1B4ndaajO3zjmYJoSIbXFZI6V7XkWCmsll809n57nLBE4JhKnHb17'
    '4LfDiLRqJW1UysQKhquqUm9EAiIW6MJLhFSv4bYgekMlryg5oOJw3LawO/QkEW5WawFcYtNTj4+xFUHi1SpS2Pkbcv4fHJ5GxJ7u'
    '1Fhaeigd9BdaU7VceK30Oy8HqgOYfWUEshIhTYWV7wLZDJ6wA3i4O2KaEYmQBtUEiNAFWCyrmIIW8j09+CmQDBtxwXA33I7QuBG1'
    'BFVtTQlMiqCII95DPu1od0x64k1WaXlHRCmrpSEc6hr+RchSDAeTCk9ksSS3ApqJnXYIALuI2cqLRVXAy4j99/xaRDfUNy4FJbPJ'
    'EPLPUJ0qL0RHtFxBVH118DprIrqGxaZXG9Dtg8HSxbubZa7BetMdlF2NoPWf712fej5O8U3ZaNjpL1gVIb9gelZ0TepSAyBwJAp4'
    'uSFy0Di/mV1DJsLMKHeKJiu/720il3SOpZAQgXtjdr4pNGEs6UU2N6peXMfbug8jaWfnVZyIGGUNIRrQz76NsYmeAxq6B51wxwPk'
    'k/aLcfLvHr14bIGAQI6AYp6PWEZUfXF4oW7ynrY60exKTR9tFYwNipbYF8JZzmLR5vldm0g9VmrGnMgE7+bZAvsu4h8qg164vkDt'
    'BXqXkrdah0BRIwfwIJ6QqgpPcTxvI1YwB8RoFUC09Vqi7pR2608sKGue97cGobmHmTZZPVA+aacs8aSp8HcGT1J8+/l4H/vCZQH/'
    'LBzaxJieCCZj1csNRG78AFFIeiDCMfUUUV8LJVmCHlLtSkwNVisRKkvzSb04h7BxpfRy9oigdpV/TrFRyAvSt+A1T62kRFCGy1sr'
    'm2IKOHIMvID6ERsO2+J+mcBERaEFWAWigeB3j7kmhTktE5FiGZJPuqHVjLsTxU+qpGOwI81w7cR3jlh368/yPTSSNy0MW+kJrw7E'
    'fF6I26102Kfuiu6EqdXQk/NFnl1Zj9J5Nrw6J2O6THXMNnTS8qxv8IRkZ3gD8UHiljKfU6toDtKgKN5hTkaJfEwSvWYbkchb0ttW'
    'tGAm4lg0wAOnEAOrtjR5yvSs+mHoWL7dytobLWZzRe9OlxOtisPDhamVoimK5ilHz5BMfNpyYIthrhXdtke0IhwN/XxypZYc02Xl'
    'Kz9i8StkCac5R5oBAUw5fpxKMMmEckR9xLDWoIVt/JesdhbiKxXUwxHqbXHci/7NvCNhF4oL4FabLh1Daox2MdIW7dBQsqDVYNNY'
    'bgsEx2llXSQuNOrL7Tf1RQ2Cdo8yHce3yZkKdtmH/s+2UhMHxDthdoFO2d2PaeXf4DG8KXU5kId8FmvzjLFvGetidp4bky2Cc+EW'
    'j22kAf1DWPh+lcBGf2/HW4jK+nsCUhbS2DYcnppAX6APXEiUgd0W43ARcVeSxG4Zgo+xW9mS7bYUb8tHv5IgxWzFtgOw11zQfru+'
    'xV48C9z5YDqgjB8K5qgYj5tAYlu3YMI93fA+f+a+VJH2pfCLDoWdgSdwv1lzftL+fkwaBmw7DFmapw1cMUwq2eK31A4iwqC8qnGD'
    '/bb/IXuB1+polougwBKvwwhVgbM1HTLCunUqEZ/xXouLaYLZoX3TCYkx28IeCeE7e3XHqKFtbDscwCr7Lp0WaA/zYhLvYVvc2dOI'
    'VoGm1iSo3LpiwBGrSmBUUXDRfhDvW3PxYpPluJYMG0Ks6jXxEumE1gencG8WMHHtABd7mHDb2RVrn+1t1BBW0oSsDZmwUancCrv5'
    'cA+YE5GXKq4S3XbQbimSGXzT7js2o3jvNDpDO5a18YRGUAPZMdKsgtdkDavDU71yjDqrJlS1/UxV+n85oV5D1KtHTBu3kdRhfXJh'
    'cVDfw30GT6G+xp+ADfAv6f1mbdlQXBO2VbKdy7vEj42Rl7KtiMiz08XOq1waWsKBLJiK9U0sT0auURYw7YM4W/uso/KHNO/FKHH8'
    'ATnb2J1W6tzB+dinCmrRK/CIZYJ1AgZFsHwaGwpYI1BBUTvb0gWXI4wq5baR+HhreccrZWandwJ10LiY7oUBH8moDkFF6vphRMwE'
    'PHZKFhySuqw91J16AWLIZ1T1q06F+XYYrvYf1BCHppW91J1u/UpPLIUepue0A8QX4Chz3TrLyNXS8grjIuFiPselQQzJ9qRqBcRr'
    'YUlBK6DqL8/ocIZFNNgkMfaYsSPdgpSUaCcgKIAEzW0UAyP84YUCKCqNqv81X8wqr35jHV7ILRlYTroixcb7mGCrR9B31dU+qnb2'
    'GNNteJ9JHQzgOwW5nFilnHgw6XO/oI97Fg9qtpTnnOyrfOiNScXRmVv1Egpi4mnWerbZ+6Tkq8uWbl2PlGbkMlAVyNuqD6Vsm3Ml'
    '7rqd3VRGYMyEh2kgmwMjAuqAg6S24i/7BMJ0huJoB9p1pnTbEP5pHWz7uqywmPOv2FhVG+TbJdZkwfykG3FKk4QHVF16ygtsB/Bv'
    'UeVLMFYC5NRgoZCsC1sBJ5VKGw2NoIrfCP1vOKHI/SNzQCNG6Mc+G+Qte5XDLpAZcHYD/ZhLrGjz4RsOMNUnAEwZMRrOJ3QATCub'
    'RAZ8gME/wKV1qxS2wGKZptg0JmOg2l0J1fWcYtPapowzoVTcWgUxmiNtaIll+mRREgd399d71JEV+23zVmtbp+XGK4rddXZgWsmW'
    'cd0m9e3nNvrFBdRmHYL+x2dtYKX5sNcz7kGsDX2MSkHySzlTqy08K1cl7Ptqf9VR6IQI3cNS8x/+phWpxYilZzbhKuHBsRySaE8w'
    'nEFyDSfSOZuej7IIJ6ssbHCiJNEek6qYIgRoYZbmcgY54JKOWpGhQRsf1GojD6wJiHdj+dEXNtJxEFF4LkWt9CKnmAO7WfILkNgo'
    'vw23UKJrSwqQosuWNJ54GzbV4/H2uUZH6j2Too6mz7zeacMjN4AJIwCouN6QLco8bt9ytqLa/tpI9zp+KRZuxgF6205sld69TFQV'
    'TgiIvWRbRzsF/VRGx+26bWpUOX1QnGDb3xn4H6332m3C2/NdUq+HPNXgO8VjednUizfgADiqPIEVQZUipXa6vb7llmZi0XNQA3bP'
    '5DlUZWQqbSOYIAJRd0MFwmH00hbJ1WpdlVAoKmshyafz5V1c6vmRlyTV+OX7nj3F6UZDxUG42wXHEWxP+wo6MRX7Vu+IzoEFNhun'
    'EE9Feg+YS01ShTWDGYu77K5aRn63chIDSzoNcTQr9IVmuXYYaETcvnG+Jk4vyQCuiHUFw3RGkC5DQxa/rqPcSBOmDV9Y4W3LwLKS'
    'pQy/ZbtmZKeHIBr5goG1mRRSiiiieAPpuwPWn393jQgcZ434M84qilbNyBWg1J1ttuCrOfksMjfvIq+USnTXGKJUjF0gK1a9Akf/'
    '9DtOccPY/dZWxkN95jiHUIH6TMRFsN0WpaQmGE8VRVSc3hLnjQDZJzT3Ac5XAWSVdq4BPiC3vCcu4aw81U1t4m0jmNYVno6C4vPV'
    'xRUhfOatZs5/v8cWcZRr4/AlIvdg7bvTS5tNfSq4j85AciDOBc7HcjSZjs/5j0dEMJbp56YZWUE3/c7zzc2BZ5T67IbkOVyiPmsa'
    'zbDiNM1ZM/E0tDh8sEU9rUlDKJ5z4Fai7sirR1F5eEGKP+VLyOVlzGiFI0tssgo8h8rBcIyhJ+7T0FX5OUiJDjbvHrwxVuWztXX4'
    '5JXv1NHWOdCrimXKWx4ZDDGN1uD1HqDeq/guCtY+srFeUtDB0/Ta2/f67wF8QPue9Bd86SrP58RnVPiOIfNg234PBW563bBtHwUF'
    'j5oK0jHO95DBj9rGeg/1LZ2NlT3K0dnn/l40d9xttlp/Tttp4RybawnF2oqfg4rADpcQAMHGddJftDTozwOCrWfNHtZXhFJUOrEJ'
    'eV+IVDsquFgFdXDEAnQNb1Gf2MkG2U7uUkJYjCA3+rclvmUn8oneimIuh5brGo0ypY5JXbNyS1HJi/gkaYAlR5xrx9ZO2hIuTWFt'
    'k/h/C4QWB0U680w6NxtUsYaku4jpNgYBV/9PaQGxFegGIsz8Kr4SagDRsce1eoj2EAlCx3453vSEPC8MsYYCFGvuuZq7JhTwmW0H'
    'A19p/NDtZWnKuMKOaIniIq1m4yUZaL/KgGCefbb662byr/ZYvN0fCVfi66b+O+ziK7OKlOwrsfEqPEQJHL0sR3PwWgwdwQy0YbqY'
    'zslsvi2GjRPf9Ysus3SzgmA2Db1wCPO2qtDYFFDOHVJ9Y7YllhnVyjoDIfu36nCtO8gMReVN3adPo+fJIxqTSY1PTBMbYPEZ9OfP'
    '0J9pRpF14wd/S37EvOm5AcHT1kVLW3i3cWLrBXRl182WEcFYrz3xlgumAAIkBe27/fbOb1t9GQDsWvTNeA1K8CelR/UXAzTJQlvt'
    'WCIWr1bFSOb7NPDXhf6IcjVcORsbaeWIIR3D7QvkUUuV3tooitxBz3JujdJlyvHszvNcqGPzWzj+hTpbQLDA1RNXemD4z8T6cuF9'
    '4Zbw5apB/bGDAOb6natB/fi0qEkNdiyM79m2tgPTWPFigwTvWNpEiL1B5ND/LJjSn8dbOv2IahKAuY4fkMPFnmP9zPEffQaZq5L+'
    'hCETCt7h2POWnTe8IM0TLPHEl0SCSWNcMM+pUf3Ds/o/N4tmwvRc/avzpO31TXMhMvk6/qgEHkgbAzc7n1swGZxMOg8VP8eyJPh5'
    'pM+yLlT+FZj8kRhJ0ujoI0p8tkHl0hsW8ztEoGdkpYCBuJo7YUpTJPkgwLGDustsQsJVLVHIxobOp6EcPFSHXLBhybrIQRK5ixUI'
    'vQgVgVgJmxUmrs+qIbU7aLFzZJ1+dwYDhzsyG04RE7AobmM+VWM6DF6HSEvOmzIwbwrffMihvT4PJL0TMymwnD4zorcD3NQ+0z99'
    '/v037cQZ1Fxg9L12HNx22Trv6am/vjVIHjqwz1PdsseigEkcp4F8yW21FIJT+R5KRwotulmGVXNTOeUJacYko45PmdIs93z/PPHh'
    'N46Kz9mW0l6Qj0N1qYYk9T4FL3Hdd5srvHa6t/eV1x1u9ueG7bn0qTu5qi23LH0JvEAoUC006ca6astts2XPignaKFaKd+XtWB27'
    'yvi4gBPsngTrWGmB9uyfYr16mRylAuUR9g5dVE/lQ37uuSbc981tNBlp3j+uDccfb1j47+RfdfmB7i+JnsB1bAwFZDhB4d+NhKO4'
    'wVgm7hz59whO/LsWmMhzUIp5e/t3K2uFdG8Ir9CgIMbHUHtA6rFqdp2aeUYJnFhQNA86XkwRT1bJvP/OAmiG5Abb5E5cINsVwpRX'
    'Npx9uWAItLeMJuYcntW8IF5BJ03Lr9IMXHs5NmUmMxVWQr0UCwSFiQkOmPTUBjxmzAkTWNELeAVwEH2byj6LdAappxFjYj1SOaSe'
    '6R9PMei+6dcAWOFRaOuZCjXfJqT2zYwMUA3gn8q5PoL/jEeJ2ty6VP6Ngz+8habgNbCsVmSgY+1SmgH1hriXvDDLTxFBYg0oHuW9'
    'b6a892214QhlbM1jeeMYNWp++pjMHL+1TwmUkESymt3+/2EDWWXIQMxsWt1A0Wq0WYmgxDzN6Kj89J9kH3c0WFXE0LvnslC+UDdY'
    'hfU9jkNjE6rE2boS3smmdtA9bFLzZUbHGEMNmARyIaa2giG4Jp1N36bTEBU2Eeh+wrgGSZQ214ztFpD2G75zV56T3rhV/9gQ3RbU'
    'of0c4J+qWbmv7HNf+w0tDFtJaMHwwoRCHeG5ngPO1ddQgzSSligwVoFs0B1X1ac50RvSn2drTER0aJB/sGWYJ59eyS8+LTGjvKQd'
    'NXmUKwHTRy/4R4tjZMivTq/wq9WML/MHAKBE4RAF5VDq0JCQ04G1KnrkFseUkhjD/KthCEL4jiJCQwo9/IgojtdHfYFEzqm+Gazj'
    '7V+SU4fsIayQwPePZJ4L5Jzg4wpjpKdbegucWvKSalHejnvW0fgiXOWPW3/ji76dCC0TPLSypJZM1UA/qpkaOEKpfNATNqivIc3N'
    'FHrtkghrPr2HVykY37Q6THAIRe6k462VvNNg8xHIdM8prFTJ3dWiuv6I1UUejPT48NQLH017f/Qa40lx9AZ+LhwqgRWIjkSA7HXk'
    'khP8JN9NnnAk5aB6HfaaSgIDOoBRkBwnGXk5eWiuCRXzG/NJOA05YgF41RzfKvMbVkBKpwkkfgiCzVxbW0sbHEAPsPBc5Xwmww03'
    '42I/WTBsIPXPxphgI4ktg2xjSgcograHnWCaxuLBxgrinGMGRURJ8DjIV/FIYlM/R1QAsJfB76AAUEprSiLUX2cACs7gUY5MAc6S'
    'UhqHo5ePZH4VMzzDBc6kn34AMXI2g9JuOQBMvOFmHBLps+Rk0zpPJpkkdZrnDs/0XRRD6PmUEJo/oZEioVjzaKQn7o4+jFp8YFm1'
    'kHiHxNXZU0unDwQymp5u18MMXd/WcDb5prquyV+dJGEQ64qr6m+2o310f8CpA+nghx6yIqSgmI/1/C5vxLiKGoxEtMXx4mKy9JeY'
    'OkYLx5LJuIucHxeUmOqbMW4YgYNx9CyIqqB+hmhzMZVdPIhlD3yfqrS4ogkHNLg1XdK4RQUfrX4JR6AlqlScft+ilAi2oIQr++T8'
    'qy6LMZ3xl4N6QGxBvJzsVPHNAtsVBWlmfIgjaUe/KRbHCJdgTbUkeH3pQ33Ef/YD7fZ+ut1vN1Eusj/MM8prSKGY2u3DJBys030/'
    'bkUhZl+F7FKdgRvMysnW9zhatSmalT/j6D9SkxR0/p1zdOPqA/ahKwpTp7urYmTydINzfQVu+MA3ULVJ2JV3LfbL4c1tW7cRcV71'
    'DquIcWwTtCMvVhfb9m5CxfkPjpxHRL9fSYv3Csbd6GviAIQlSwAexYBiIQR/NMDjCJ49Sm5mCEKqCdmMXFrQjbzCsXQV2pjfZsMl'
    'ZYldNh7NjmV4nKv531XXqK3dxKCAXnLGaz5uigZhB9YD/RrFx2Azu8l/nersAUh5gE2Es8NT8nYOr5TQHiQqltyjyLHWXhkffJ6R'
    'coW9iXHY0ts2HZtdFFm5/KbwYQLGUYtV7qtcrOqaIk2Hwo1WsOpwuOEjqbjIGJB30Ns9AsptUW0uHrwnXXMRvFPDJWux/Aq+1y4N'
    'CL7PWSCiBgWBTodxa4ffXVmVAQC+5iXtNsdeS6QiRb+lPDmWa/oBcQ2GXgf4q3sDuwXy4tuBk7pD3Gez6QoDimaCx5zvKqHT25z3'
    'AXwuivVNCkTFWDTFF9rP+LBu2ADFuS9Tl2jcuPDaV6rjktW5SglM8uAGX2wkXmChEacxX2hzZ6VlRKJviHt3Mc9I9haMPlflEZ+9'
    'UWNVjLyC1+/AE692RYNrYpJ8LQxlos7okDJdmAQ+fz0tiZGxLNBut+8F3cU6au5tvWBDHwCez4CHMmBEe08VcXCayjVh7x15VJC/'
    'Q2tWY9bRHYa3GIl+NX75bpEAv1+/J93wq3/02P/oaT141XIXvnowykp4horcUATmFw+D7VpTOMLLtkiq8YtPQSvLyr9PHT7VmHuY'
    'NhAuSirNV0+lsTtPh0XJETYl1jhAUS0hKcS3utvP4wPhvyp2p0LhdeS7KLPBLOzrmdT8ykeXX1j2aFeW87+/5s183YjRstxrUlGA'
    'oa9o87U9OUI0Seyluj5X6bYooTqdYlxSVidJDV8iZ9vUk5bm6vymU6wt8bxna22TIRulct5yCKabblrOSDCQfgdOOdcqrOvoBB/y'
    'gg2RSx2g3PCpIuee11Xt2z6lm1yQ9pPyg9CCxFr9vHG18WVjxgsZd7HkG9fzjREFwoGJ8O0cSE3cozRIdFCdzjKznEK3fVAun1HS'
    '1tl++NOCPyGiVe704JMkOEdNMiKxAckuQfLTnZYM23T70/Hb3imnVeZrNuBdIhX7k5xHvnJ+f3F+z5zf1HHn8XruPNBg1COEhu7r'
    '7tEp7rWQNk+UZfrk192jox5Mt7sHvS7fdqLThMqsCxpVigx6NVKYtUKNOsjjJFmREi3ahkXSTlam01SVhXb4PBgttVilianafm56'
    'vy7f/7Ud1eiOiMD6dDMREF1UOxADrGhTK1/U0OiXDRDWJ/h8bNJ/z2mq6BsFqMqhSeVgDlr0HM2NuWGQK3vJ1KsP2bF4UalLAPAf'
    'JYMVbEU75lIQnWlaTSCn5ZdF+evuAc7EH344OA1mm9AU9K7zWExboAYj9lUDMnQAs743So9Qhkahy3RYmnMYWThuWicPFfWFpg8h'
    '3mnVjKe0I74yybwZzBizinsEHhbaUjkb4B3Z6PdxIc4/6JYWuumpFf34o84QPBg05xdRaXPqJ59MGzalx5CufxRGredGziidOBkJ'
    '75891uU55sxVfrGeLzyezzSPQ8By+5j+wReP6Qd1dZh+VFekWYbsQKMUjcJCVPOqVDAIveBsoXsnIRsv2Yn2MOKw3pTklfqJUBTy'
    '1wXWqvwnxvYk1+xQLoMIJcW4lRFHoZiG6lInZtsUmvLatGlO/K78xJehEYfm+UATZBEZebq0sjGoliiLqZ3/o+5xd9+d/qk79wqx'
    'HR/dUsdkZtDCRvKYUq8SylmmyqxOW+YDcXBpL+mT8ZrsYzAV0LEEmwKYs7uowwk4VvCnabXT+ksutJLMvbY3G5HkKXODTxRPofwF'
    'cFpo0uGnwD+t9jjqPe5wDVlzK8SpJ6gTOJX3FmYcg1HIXZSTP1bHYfT6RStGxm0uUct5sXQuhOH0h/64+G97Icz/0WBgEiFfgjOm'
    'b9IB7eDFAs/Ct4sR1+oaVnqlKp27lV6ZSjrDD3EXJiROHBs7yR4lId+qiwn1gtKeGpHAZuOxRdw9y0c6yu02NWHvPdTg5WJYK1JT'
    '51b0+Bs6oCuoiyo9OwJ2/V26726/uAXl5Ouz8XqPZVadsolND9LNSm53uswnc05v5mSWG5FbKqMMq4iFkKNYet+QTbYtg7riW1nG'
    'sXsflw6ibBYPdMbAe0Y5teYVpAD20Xb/fjBcelvyg9m/YvDd5y9/WJH4i8Ct3JxVWw1bM3c+Pb9LJcFLKEJ/64okH52HZiWAuK30'
    'rzinh/uKpawrJWIpjNYFLUG0FrDs3SL2pkbeyDzIKtslZ8qz0pgV/1d1I7x2qEm67RrKVPlxnujcnnwQiUDnlG7MkOr5HXUybyZR'
    '2jmuVCLWTGVLNWuU08/JZuvY8UFwiGwbpXp/CHaPe8nvUYtXs0EYF9KYr1x96lBrKU5KuyMn9bqvHlf3VeKQ8SpJKaRiUK9gxAhI'
    'ziJYt6BWCUpKUFFAGFc1xMdWTvpZS0pSQfL+qso/PaAIdUO4ONrI2mNtTjs+08tUqtibgg5EFKXoQU17qqUERURiCGDCZPJB3EtO'
    'HqHM3kG6vIQB4OIySBdMZl+FemFp7ublZI/1pSaTPFa5BSSLB00CHB3VY5jSfHYTP9eZAG3VNlWU9J1nJWez/J9mVxfu0Q4zXEvo'
    'jkQUxPWp0a9O2XBBtlF1zI3LKtokwVSvAXeMwizdN4ZJtZlbXlh6NtM6JW+hp4j+edWJJKUdfAY/JDrpwBdJOeUAR6zMFP356yG+'
    'ubJHP/2LjPOCTCmzZtLUlGOZ5DfRYG1XlvVhkvnwmDvmTLE6YEpbh9zQenK0t3uqjFzqRpYBL37voni5eZGvaVl1vtsFpzUVVcU0'
    'tos7vl0FyRwYdyt3ave0rDgjXs8kVruoJTiwZPvJPQmP6dM1OEnzNbV+/52nfqfhvttBoGTRRKj4dJOz34Ghsvs7NyC5+OA51fdX'
    'O3ee0HXVKa6rNvedGPmWL9Pm7Ak4JYD5N6qWGSZF9eKDvSkHL0LkmfhhshGr62IZcizBW3TQ0Lvlu2VubjHn4SeU18e/e9t4Si+L'
    '+jebv4tyypDk42XvNPRi0EGEqahEv6tThwstKMUj7U9mnctC5xbNL4KcoTQqAyIkKVXcR5ApLViqX0reCoAa7kjX3mBJq+mo3dGk'
    'Au45iYYJWda5DFSewXpqA2VuusSVItwu9RnMLGq4LJ29Zwz+p+ilO0556fu+hnbgcvdTrNv23WByEE4PjQ/ObXOej0gfoqMePXgH'
    'KRUKE7/oA4aUF5m+O6sAl8fzFstLwMSo15OD0OGU5pc6iZQ2Tdj76oMDYmhATuwTr61dhxSuNwWqpWmnvl4fnaGk8NKhmf6FxfGh'
    '8c4qWtzPmpBfy83JiS6GDR5kKqgitSzOGvzH9qPJMcmpwwzykxV1vPGru2t02iU9a3pSnzlV1E3ifmoUU0lC+vUhDIqR6PU71CEF'
    'Si+1HfnkfunoxKLFlNqOUfsXlCO3HZ01AUPHcuqefjjuuYlP+AwbE6Kbm6QRBZiPVIWEik5BrnPKAtKiWn5YxDndjGVPenjVJLTi'
    'wXrqQIgmYnu0w1s3D9Dz4xiXORzelBLKwUPICmzO1v7zQf2cbSobgMdhuAozmQZi1BimbH+fBvUwH3eByIpCprb+1qCWoqwyYBz6'
    'rq0aPulC9FabX/2TbGxET7532JlR/bO5IIGZOLnFcJIIMjdcpFLWix3XTVL2jfwfiUoE0JgoQ44LU+4VyjtBoQuxbmmdyV/3E4qa'
    'CJZYDlLpqe2x0oB04ku+zs3/pL64uSHs6TTF4HUTRoOi/4mYZOzunGWgZu7m/HGebf7BHHR+FQbrVAhvAUH241+PThuvMoZTRt9k'
    'DP1zKZcKL1kifgXh8qE6Wgumq9F0vRMEzZN7DL6xvT8s6SqJsYf8vq9PtIr3bhelj/9Qw+YW32yZy+j4uXd0uPOOX6kXuPf01Caq'
    'NKeqrxfkzhcdHO6S4dJN/P/JWGns7XePSvTn1dXbrJM4jnU4F/Vu+1qfC1elgSn65ZcVhn2Pvv40yKYDM4Gtic/qsL++40D/K3AD'
    '2K7RQd0YDs9zmxK5ZtKgz32nUSdT2gq0sWymqqnogRVYdREWdC32AThaA+Ie4aoDW6ETI0ZjCLVLnafNzx5mNlltZdZqX3DU6t4T'
    'Kax/6LtAHd+lb6AJskrYnBJA3PGHA9HDGFWhkuUI7iQbVinxlHw+ww0G4T0CtaobkdLsWFQ+Oe0dmcOvCAKCTEEASVKg1US3/wYt'
    'KAkc96yYomSw2+Ibb2LcbLb/4Sh9c9zdYTuigRimFZO6ZPxgqKwqBi2pOw8JhnyS+4YcmDrijZOKsij7pzrgRJFpkctT+Ugar4i9'
    '4798hchhkaFKJICd03EWuFPLwuaXaEBnYMSxsSmnrTDyD71pw7Ay/U1SoYJaJJoV9DrM7rZ/6+2+fQeu2dvp/qF9+XAvpZNFKKpf'
    'sGVdtyCcPBWYbtCmqqzHwOe1nEXKcW62f9i12m9hlDnh1+ZibD/Tqtxb7QWPTsAMqEpMExMY2+gVdDBLMp3GuDquKmn8N5zCHofF'
    'XR5so5Sy606xROUc2fIoxC9SC1TbbG/xAfV/gFZxTF2f4n4mwa24oUYW0pyuLNCtJ2bchtmfrcnKAAFsMoOdVeY3tWwfcqiCI/nY'
    'lqISpMps7y8NmNZ/7+TDfs9LKirBt3ZXUiX1hpvf0qG02Ozf9ezRAsAUeOBuBynNV9OTMZgu8YxBOxtX2QWiYTf4zhl1p3UoDzi9'
    'Ezir7azEwx1Zj27qlqNZSPNt7gzdVpeu66WCQ9fwMdbuGzLbU8P21Vd3bVnCN2dHzQpasb/MaFRhLVk2q6rI14ZaimJ0MfbArwVq'
    'gJV0VDHEewM/d7WCVghSBfnaWT7d2lBQS0duWUW2TcW17OQWb+6vmeW/OcTVFHZ9yZe4cq4BNcmWWEO5RzYJusCXUsRSDC1Nv6pG'
    '24LqektFFJglqC5Zl2Vnv8rzwONZ5IpJiXY9uUtR5J+GYDqNEiNHSc75dlVLQWEBTQcdTS9egTqKDEl0pON8ZYye/Y6mDL7iTk91'
    'R5OBeWt4kZ32xqbcKe04k+6IrrIsCU9M2c6EgUfCAb+czr0FD16EnBt0piduKulPuBaLmCVan7iZMc0rtaUwRdQAsQo4KNr0lxah'
    '4H1C3B9dNtX0ldIuKMeWcRXRu9IaHeuSmd1kRagyiePqxCZiUm3Hk2LC3QcNV0krXXPqZ64PhbUwoEWbZi+vx2OyzeaLaROHcVre'
    '5lZWlKFtauBfhKT6FIDx7HLu51nlfRW7LJX5SazC1MGQ41Ma+FRS+NH3Pop3xHoXCqYhoyEdKcjP5+U6N6Cb0kNQZS0EuU6PopG1'
    '07xYkSbkIWHrFy0tZ2nPZrNcZopVTXc3GNFMHXMicE+jhoMIip5Ds3DjDiVDoFKJ56BtKIqgHPqR2s4H5VSEQsm3cSONzKSYy7WF'
    'JR/08UwYz3wB+e0x3ZK1t3u0opcYyspmdff4hGS8cvvmszYcO8ELepZSMlhtA6pZu3QiPEMzyQOLhH40llBkETgUPN5jfMnsJ9c6'
    'fFIvXSoodeN00NgTxMe+TXsf4ejgpXLSlHFNZLtsXkk+f8saIRZbrtmQUYRV+5xO4enqG2F7DZlL8rHkf3Wl76DfDW2prV4Y7RO6'
    'QpybfjLa4AfGH93xR78XlGM6x68n7S3gZINLQhbAsZ0n7U1KoFlKZGdzuhGlVQZ98hSGlp000Sd4ViiFIcK9jvVyXJHOROOtJbh4'
    'ahG5Ef1AXvem7CsN+1OdKjwebZbjyo1vBaGaHZWPB3n7Lg7/hvSBlpNH0OFJ92PvMYToSFs1qH7TkEKOT9PT3f0etPGT3snJLlz3'
    'O90jMgl+C2A+SxGYGK3puokIT6Df8Rm7gu6pxUVDiFdEvM1InXqQS7o5gqcddZe0iuU0JpSbHG6IK7pkVEVVIySolOth2YJDskt2'
    'QSfacWhPxNq2JzoFl+o6yPGGEG6m+gpeGy6K/Y2PF7FVhS7tLfTBcjJHlSFRaEmG1jovwWcuF7tPVKiLGr5uKrkqfPXY2HW0WhzQ'
    '01/+lYfQGb81dsMuDjL2NUmEDXmTfIOhX4yHIcY80a1IK95aXcZJZMTbxuqSOsHRwD9G3cgVKT1OBUoEt6Nz0PuUimb9zdYvT4iz'
    'M3tEKnn6l87c853mxBcJA8QYK+GKCvucqoZG3lI4CFc+1UvqqxSo+llrGw3r0Kqj460VX7UO6hgSVyqeTUpXAw+0Goqt3YqMRyJZ'
    'vdjJ13QTkRakF8fa/fddepPCNjFNzXp2PAhKW9MyJ8+8eyK7iW0By86FRyoPPVvgOk2iGH2SM+3I+Q5GZXl/6ENZpVWpVbaKx9RC'
    'f0ZkvVf7L3Zb/GQEEk1i6pkQcaBEk62Qnd8X2QpbVm21lBFcc+6Qg1KgyB3qMoTOPYaLh+nnEbTjUEHd1GYLOv3w+uCYtUzhbzZt'
    'BZT4TZa5Jq/bN1jnVg3mX7PR1aa00YumDt+hGXA73mtNMK6CbEYRYEbXPXAuAKQdA+GKjXDckO+iarghkD+11d0jTSuFwd87piZ7'
    'pK0Q+ER4KelOKgzEK5aWOsK0YhUFzq61wX8DVokLbg=='
)

In [ ]:
import os
import sys
import io
import gc
import json
import time
import zlib
import base64
import contextlib

import torch

# Folders searched for inputs. On Kaggle this is /kaggle/input (the env var is only for local tests).
INPUT_ROOTS = os.environ.get("PCR_INPUT_ROOTS", "/kaggle/input").split(":")


def _walk_all():
    for root in INPUT_ROOTS:
        if not os.path.isdir(root):
            continue
        for dir_path, dir_names, file_names in os.walk(root):
            yield dir_path, file_names


def _prefer_final(paths):
    paths = sorted(paths)
    for p in paths:
        if "final" in p.lower():
            return p
    if len(paths) > 0:
        return paths[0]
    return None


def find_assets():
    # Locate the data bundle, the two Phase-1 model folders and the two model code folders.
    # Model 1 and Model 4 both have a file called legal_gqa_model.py, so the folder that also holds moe.py is Model 4.
    assets = {"bundle": None, "m1_dir": None, "m1_py": None, "m4_dir": None, "m4_py": None}
    m1_dirs = []
    m4_dirs = []
    for dir_path, file_names in _walk_all():
        low = dir_path.lower()
        if "tokenized_pcr.pkl" in file_names and assets["bundle"] is None:
            assets["bundle"] = os.path.join(dir_path, "tokenized_pcr.pkl")
        if "config.json" in file_names and "checkpoint" not in low:
            model_type = ""
            try:
                with open(os.path.join(dir_path, "config.json")) as f:
                    model_type = str(json.load(f).get("model_type", ""))
            except Exception:
                model_type = ""
            if model_type == "legal_gqa_mha_moe" or "gqamoe" in low:
                m4_dirs.append(dir_path)
            elif "legal-gqa" in low or "legal_gqa" in model_type:
                m1_dirs.append(dir_path)
        if "model.py" in file_names and "moe.py" in file_names and "legal_gqa_model.py" in file_names:
            if assets["m4_py"] is None:
                assets["m4_py"] = os.path.join(dir_path, "model.py")
        elif "legal_gqa_model.py" in file_names and "moe.py" not in file_names:
            if assets["m1_py"] is None:
                assets["m1_py"] = os.path.join(dir_path, "legal_gqa_model.py")
    assets["m1_dir"] = _prefer_final(m1_dirs)
    assets["m4_dir"] = _prefer_final(m4_dirs)
    return assets


def find_run_dirs():
    # Finished PCR runs: a folder holding run_summary.json and adapter_best.pt.
    found = {}
    for dir_path, file_names in _walk_all():
        if "run_summary.json" not in file_names or "adapter_best.pt" not in file_names:
            continue
        try:
            with open(os.path.join(dir_path, "run_summary.json")) as f:
                summ = json.load(f)
        except Exception:
            continue
        if "pooling" not in summ or "adapter" not in summ or "model" not in summ:
            continue
        model = "m4" if "Model 4" in str(summ["model"]) else "m1"
        adapter = "lorma" if str(summ["adapter"]).lower() == "lorma" else "lora"
        key = model + "_" + adapter
        if summ.get("kd"):
            key = key + "_kd"
        if key not in found:
            found[key] = []
        found[key].append(dir_path)
    runs = {}
    for key in sorted(found):
        paths = sorted(found[key])
        runs[key] = paths[0]
        if len(paths) > 1:
            print("note: several runs match %s, using %s (others: %s)" % (key, paths[0], paths[1:]))
    return runs


@contextlib.contextmanager
def _env(values):
    old = {}
    for k in values:
        old[k] = os.environ.get(k)
        os.environ[k] = str(values[k])
    try:
        yield
    finally:
        for k in old:
            if old[k] is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = old[k]


def _apply_settings(ns, summ):
    # The run replays with exactly the settings it was trained with (read from its run_summary.json).
    ns["RUN_TRAIN"] = False
    ns["RESUME"] = False
    ns["RUN_TEST"] = False
    ns["POOLING"] = summ["pooling"]
    ns["DOC_VIEW"] = summ["doc_view"]
    ns["HEAD_CHUNKS"] = int(summ["head_chunks"])
    ns["TAIL_CHUNKS"] = int(summ["tail_chunks"])
    ns["CHUNK_LEN"] = int(summ["chunk_len"])
    ns["CONTENT_LEN"] = ns["CHUNK_LEN"] - 1
    ns["USE_PROJ"] = int(summ.get("projection_params", 0)) > 0
    prefix = "LORMA_" if str(summ["adapter"]).lower() == "lorma" else "LORA_"
    settings = summ.get("adapter_settings", {})
    for k in settings:
        ns[prefix + k.upper()] = settings[k]


def _load_cells(key):
    blob = _SOURCES_B64[key]
    return json.loads(zlib.decompress(base64.b64decode(blob)).decode())


def _free(ns):
    ns.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    sys.modules.pop("legal_moe_model", None)


def embed_run(key, run_dir, splits, assets):
    # Rebuild one finished run (frozen Phase-1 model + its adapter + projection) and embed the given splits.
    with open(os.path.join(run_dir, "run_summary.json")) as f:
        summ = json.load(f)
    src_key = key[:-3] if key.endswith("_kd") else key
    model = src_key.split("_")[0]
    if assets[model + "_dir"] is None or assets[model + "_py"] is None:
        raise FileNotFoundError("Phase-1 %s folder or code not found among the inputs (needed for %s)" % (model, key))
    cells = _load_cells(src_key)
    env = {"PCR_OUT_DIR": run_dir, "PCR_MODEL_DIR": assets[model + "_dir"], "PCR_MODEL_PY": assets[model + "_py"]}
    if assets["bundle"] is not None:
        env["PCR_DATA_DIR"] = os.path.dirname(assets["bundle"])
    ns = {"__name__": "__main__"}
    buf = io.StringIO()
    t0 = time.time()
    try:
        with _env(env):
            with contextlib.redirect_stdout(buf):
                for i, src in enumerate(cells):
                    exec(compile(src, "<%s cell %d>" % (key, i), "exec"), ns)
                    if i == 0:
                        _apply_settings(ns, summ)
                if ns.get("BEST_STATE") is None:
                    raise RuntimeError("adapter_best.pt was not loaded from " + run_dir)
                if int(ns["N_ADAPTER"]) != int(summ["adapter_params"]):
                    raise RuntimeError("adapter size %d does not match the run summary (%d) for %s" % (
                        int(ns["N_ADAPTER"]), int(summ["adapter_params"]), key))
    except Exception:
        print(buf.getvalue()[-3000:])
        raise
    shown = []
    for line in buf.getvalue().split("\n"):
        for word in ["document view used", "model:", "trainable", "adapter target", "loaded saved adapter"]:
            if word in line:
                shown.append(line)
                break
    for line in shown:
        print("   [%s] %s" % (key, line))
    t_load = time.time() - t0
    result = {}
    t1 = time.time()
    for split in splits:
        result[split] = ns["embed_split"](split)
    info = {"key": key, "base_params": int(ns["N_BASE"]), "adapter_params": int(ns["N_ADAPTER"]),
            "embedding_dim": int(ns["D_MODEL"]), "pooling": ns["POOLING"], "doc_view": ns["DOC_VIEW"],
            "load_seconds": round(t_load, 1), "embed_seconds": round(time.time() - t1, 1)}
    _free(ns)
    return result, info


def make_scoring_ns(out_dir):
    # Scoring functions (cosine, CSLS, dual softmax, BM25, F1) taken from the same code the runs used. No model is loaded.
    cells = _load_cells("m1_lora")
    order = [0, 1, 2, 5]
    env = {"PCR_OUT_DIR": out_dir, "PCR_MODEL_DIR": "unused", "PCR_MODEL_PY": "unused"}
    assets = find_assets()
    if assets["bundle"] is not None:
        env["PCR_DATA_DIR"] = os.path.dirname(assets["bundle"])
    ns = {"__name__": "__main__"}
    buf = io.StringIO()
    try:
        with _env(env):
            with contextlib.redirect_stdout(buf):
                for i in order:
                    exec(compile(cells[i], "<scoring cell %d>" % i, "exec"), ns)
    except Exception:
        print(buf.getvalue()[-3000:])
        raise
    return ns

In [ ]:
ASSETS = find_assets()
RUNS = find_run_dirs()
print("inputs found:")
for k in ASSETS:
    print("  %-7s %s" % (k, ASSETS[k]))
print("finished runs found:")
for k in RUNS:
    print("  %-10s %s" % (k, RUNS[k]))
MEMBERS = []
for key in sorted(RUNS):
    if USE_ONLY is not None and key not in USE_ONLY:
        continue
    model = key.split("_")[0]
    if ASSETS[model + "_dir"] is None or ASSETS[model + "_py"] is None:
        print("skipping %s: the Phase-1 %s folder / code is not attached" % (key, model))
        continue
    MEMBERS.append(key)
if len(MEMBERS) < 2:
    raise RuntimeError("need at least two finished runs to ensemble, found: " + str(MEMBERS))
print("ensemble members:", MEMBERS)

SC = make_scoring_ns(OUT_DIR)
for name in ["make_scores", "apply_norm", "zscore_rows", "top_ids", "micro_prf", "plain_f1", "bm25_scores", "recall_at"]:
    globals()[name] = SC[name]
B = SC["B"]
GOLD = SC["GOLD"]
NORM_GRID = SC["NORM_GRID"]
HYBRID_LAMBDAS = SC["HYBRID_LAMBDAS"]
NEG_INF = float("-inf")
import torch
print("dev queries:", len(B["query_ids"]["dev"]), "| test queries:", len(B["query_ids"]["test"]))

## Embed dev and test with every member
Each run is rebuilt with the settings stored in its own `run_summary.json` (pooling, document view, adapter size) and checked against its adapter size.

In [ ]:
EMB = {}
INFO = {}
IND_F1 = {}
for key in MEMBERS:
    t0 = time.time()
    result, info = embed_run(key, RUNS[key], ["dev", "test"], ASSETS)
    EMB[key] = result
    INFO[key] = info
    d_qids, d_pids, d_Qe, d_Pe = result["dev"]
    p, r, f1 = plain_f1(d_Qe, d_Pe, d_qids, d_pids, GOLD["dev"])
    IND_F1[key] = f1
    print("%-10s dev Micro-F1@%d (plain cosine) = %.4f (P %.4f, R %.4f) | %.0fs" % (key, TOP_K, f1, p, r, time.time() - t0))

# all members must use the same query and pool order
ref = MEMBERS[0]
for key in MEMBERS:
    for split in ["dev", "test"]:
        if list(EMB[key][split][0]) != list(EMB[ref][split][0]) or list(EMB[key][split][1]) != list(EMB[ref][split][1]):
            raise RuntimeError("member %s has a different %s query/pool order" % (key, split))
print("all members share the same query and pool order")
print("note: dev is used to pick weights and options, so ensemble dev F1 is slightly optimistic. If a member was trained with TRAIN_ON_DEV=True the dev numbers are not a fair estimate.")

## Choose the combination on dev

In [ ]:
def combined(split, center, weights):
    total = None
    wsum = 0.0
    sigmas = []
    for key in weights:
        w = weights[key]
        q_ids, p_ids, Qe, Pe = EMB[key][split]
        S0 = make_scores(Qe, Pe, q_ids, p_ids, center)
        finite = torch.isfinite(S0)
        sigmas.append(float(S0[finite].std()))
        Z = zscore_rows(S0)
        if total is None:
            total = Z * w
        else:
            total = total + Z * w
        wsum += w
    sigma = sum(sigmas) / len(sigmas)
    return total / wsum * sigma


CANDS = []
CANDS.append(("equal_all", {k: 1.0 for k in MEMBERS}))
weighted = {}
for k in MEMBERS:
    weighted[k] = max(IND_F1[k], 1e-3)
CANDS.append(("f1_weighted_all", weighted))
if len(MEMBERS) > 2:
    for drop in MEMBERS:
        CANDS.append(("without_" + drop, {k: 1.0 for k in MEMBERS if k != drop}))

d_qids = EMB[ref]["dev"][0]
d_pids = EMB[ref]["dev"][1]
dev_gold = GOLD["dev"]
dev_bm25 = bm25_scores("dev", d_qids, d_pids)
dev_bm25_z = zscore_rows(dev_bm25)

ROWS = []
order = 0
t0 = time.time()
for cname, weights in CANDS:
    for center in [False, True]:
        E = combined("dev", center, weights)
        for norm, param in NORM_GRID:
            S1 = zscore_rows(apply_norm(E, norm, param))
            for lam in HYBRID_LAMBDAS:
                if lam == 0.0:
                    S2 = S1
                else:
                    S2 = S1 + lam * dev_bm25_z
                    S2 = S2.masked_fill(~torch.isfinite(S1), NEG_INF)
                lists = top_ids(S2, d_pids, TOP_K)
                p, r, f1 = micro_prf(lists, d_qids, dev_gold, TOP_K)
                ROWS.append({"order": order, "weights": cname, "center": center, "norm": norm, "param": param,
                             "lambda": lam, "precision": p, "recall": r, "f1": f1})
                order += 1
ROWS.sort(key=lambda d: (-d["f1"], d["order"]))
print("%d combinations tried in %.0fs" % (len(ROWS), time.time() - t0))
print("single runs (plain cosine, dev):", {k: round(IND_F1[k], 4) for k in MEMBERS})
for row in ROWS:
    if row["weights"] == "equal_all" and (not row["center"]) and row["norm"] == "none" and row["lambda"] == 0.0:
        print("equal-weight ensemble, plain: dev F1 %.4f" % row["f1"])
        EQUAL_PLAIN = row["f1"]
print("top 10 combinations on dev:")
for row in ROWS[:10]:
    print("  %-22s center=%-5s norm=%-4s param=%-5s bm25=%-4s -> F1 %.4f (P %.4f, R %.4f)" % (
        row["weights"], row["center"], row["norm"], row["param"], row["lambda"], row["f1"], row["precision"], row["recall"]))
CHOSEN = ROWS[0]
NEURAL_ONLY = None
for row in ROWS:
    if row["lambda"] == 0.0:
        NEURAL_ONLY = row
        break
print("chosen:", {k: CHOSEN[k] for k in ["weights", "center", "norm", "param", "lambda"]}, "-> dev F1 %.4f" % CHOSEN["f1"])
print("best neural-only:", {k: NEURAL_ONLY[k] for k in ["weights", "center", "norm", "param"]}, "-> dev F1 %.4f" % NEURAL_ONLY["f1"])
with open(os.path.join(OUT_DIR, "dev_sweep.json"), "w") as f:
    json.dump(ROWS, f)

## Test predictions
Exactly 7 unique ids per test query, only from the test pool, best first, never the query itself.

In [ ]:
PRED_PATH = os.path.join(OUT_DIR, "prediction.json")
weights_by_name = {}
for cname, weights in CANDS:
    weights_by_name[cname] = weights

if RUN_TEST and "test" in B["splits"]:
    t_qids = EMB[ref]["test"][0]
    t_pids = EMB[ref]["test"][1]
    test_bm25_z = zscore_rows(bm25_scores("test", t_qids, t_pids))

    def predict(row):
        E = combined("test", row["center"], weights_by_name[row["weights"]])
        S1 = zscore_rows(apply_norm(E, row["norm"], row["param"]))
        if row["lambda"] == 0.0:
            S2 = S1
        else:
            S2 = S1 + row["lambda"] * test_bm25_z
            S2 = S2.masked_fill(~torch.isfinite(S1), NEG_INF)
        return top_ids(S2, t_pids, TOP_K)

    def build_prediction(lists):
        pred = {}
        pool_set = set(t_pids)
        for r, q in enumerate(t_qids):
            ranked = lists[r]
            if len(ranked) != TOP_K or len(set(ranked)) != TOP_K:
                raise RuntimeError("bad ranking for query " + q)
            for c in ranked:
                if c not in pool_set:
                    raise RuntimeError("candidate outside the test pool for query " + q)
                if c == q:
                    raise RuntimeError("query retrieved itself: " + q)
            pred[q] = ranked
        if len(pred) != len(B["query_ids"]["test"]):
            raise RuntimeError("missing test queries")
        return pred

    prediction = build_prediction(predict(CHOSEN))
    with open(PRED_PATH, "w") as f:
        json.dump(prediction, f)
    with open(os.path.join(OUT_DIR, "prediction_neural_only.json"), "w") as f:
        json.dump(build_prediction(predict(NEURAL_ONLY)), f)
    zip_path = os.path.join(OUT_DIR, "submission.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(PRED_PATH, "prediction.json")
    print("wrote prediction.json for %d test queries (members %s, options %s)" % (len(prediction), MEMBERS, {k: CHOSEN[k] for k in ["weights", "center", "norm", "param", "lambda"]}))
else:
    print("RUN_TEST is off or there is no test split.")

## Run summary (for the report)

In [ ]:
summary = {
    "members": MEMBERS,
    "member_info": INFO,
    "single_run_dev_f1_plain_cosine": IND_F1,
    "weightings_tried": [c[0] for c in CANDS],
    "chosen": {k: CHOSEN[k] for k in ["weights", "center", "norm", "param", "lambda", "f1", "precision", "recall"]},
    "chosen_member_weights": weights_by_name[CHOSEN["weights"]],
    "best_neural_only": {k: NEURAL_ONLY[k] for k in ["weights", "center", "norm", "param", "lambda", "f1"]},
    "top_rows": ROWS[:15],
}
with open(os.path.join(OUT_DIR, "run_summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(json.dumps(summary, indent=2, default=str)[:2500])
for name in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, name)
    if os.path.isfile(full):
        print("  %-30s %.2f MB" % (name, os.path.getsize(full) / 1e6))